# <center><u>**CERT x466-003**</u>: Data Storytelling<br>AI Job Market Data Exploration and Storytelling</center>

<figure style="text-align: center;">
    <img src="notebook_images/ai_jobMarket01.png" alt="AI Job Market">
</figure>

---

---

---

## DOWNLOAD AND UNZIP DATASET FROM KAGGLE

In [1]:
from dotenv import load_dotenv
load_dotenv()

import kaggle
kaggle.api.dataset_download_files('shree0910/ai-and-data-science-job-market-dataset-20202026', path='./data', unzip=True)

Dataset URL: https://www.kaggle.com/datasets/shree0910/ai-and-data-science-job-market-dataset-20202026


---

## GPU ACCELERATION

In [1]:
%load_ext cudf.pandas
# NOTE: cuml.accel is disabled — RAPIDS cu12 JIT compilation fails against
# the system's CUDA 13.1 headers (/opt/cuda/include/cuda_fp8.hpp).
# cuml can still be used via direct imports (e.g., from cuml.ensemble import ...).
# cudf.pandas acceleration works fine for dataframe operations.

# --- Preload CUDA shared libs for TensorFlow + PyTorch GPU support ---
# TF 2.21 needs cu12 libs; PyTorch cu130 needs cu13 libs.
# Both are pip-installed under nvidia/*/lib but the Jupyter process
# won't pick them up via LD_LIBRARY_PATH alone (already running).
# We preload them with ctypes so dlopen() finds them in-process.
import os, site, ctypes

_nv = os.path.join(site.getsitepackages()[0], "nvidia")

# Map of (soname, subdirectory) — cu12 for TensorFlow, cu13 for PyTorch
# NOTE: nvrtc-builtins must be loaded BEFORE libnvrtc, because libnvrtc
# tries to dlopen the builtins lib at load time.
_libs_to_preload = [
    # TensorFlow cu12 libs
    ("cuda_runtime/lib/libcudart.so.12", "TF"),
    ("cublas/lib/libcublas.so.12", "TF"),
    ("cublas/lib/libcublasLt.so.12", "TF"),
    ("cufft/lib/libcufft.so.11", "TF"),
    ("cusolver/lib/libcusolver.so.11", "TF"),
    ("cusparse/lib/libcusparse.so.12", "TF"),
    ("cuda_nvrtc/lib/libnvrtc-builtins.so.12", "TF"),
    ("cuda_nvrtc/lib/libnvrtc.so.12", "TF"),
    ("nvjitlink/lib/libnvJitLink.so.12", "TF"),
    ("cudnn/lib/libcudnn.so.9", "TF+PT"),
    # PyTorch cu13 libs (PyTorch loads most via torch itself,
    # but preloading ensures availability)
    ("cu13/lib/libcudart.so.13", "PT"),
    ("cu13/lib/libcublas.so.13", "PT"),
    ("cu13/lib/libnvrtc-builtins.so.13.0", "PT"),
    ("cu13/lib/libnvrtc.so.13", "PT"),
]

_loaded = []
for _rel, _tag in _libs_to_preload:
    _path = os.path.join(_nv, _rel)
    if os.path.exists(_path):
        try:
            ctypes.CDLL(_path, mode=ctypes.RTLD_GLOBAL)
            _loaded.append(os.path.basename(_rel))
        except OSError:
            pass

# Also add all nvidia lib dirs to LD_LIBRARY_PATH for any remaining lookups
_lib_dirs = [os.path.join(_nv, d, "lib") for d in os.listdir(_nv)
             if os.path.isdir(os.path.join(_nv, d, "lib"))]
os.environ["LD_LIBRARY_PATH"] = ":".join(_lib_dirs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")

print(f"GPU acceleration configured — preloaded {len(_loaded)} CUDA libs")

GPU acceleration configured — preloaded 13 CUDA libs


---

## IMPORTS
- **NOTE**: run command `direnv allow` in terminal (within the virtual environment) before running imports cell

In [2]:
# General/Data Science Imports
import pandas as pd
import numpy as np
from scipy import stats as sp_stats
from scipy.spatial.distance import pdist, squareform
import warnings
from itertools import product

# Plotting Imports
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning Imports
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression

from cuml.ensemble import RandomForestClassifier as cuRFClassifier
from cuml.ensemble import RandomForestRegressor as cuRFRegressor
from cuml.linear_model import LinearRegression as cuLinearRegression

import torch
import torch.nn as nn
import torch.optim as optim

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

I0000 00:00:1775644187.654735    5173 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


---

## DATASET LOADING AND INITIAL LOOK

In [3]:
ai_df = pd.read_csv('data/ai_jobmarket_bootstrapped.csv')

In [5]:
print(ai_df.shape)
ai_df

(16888, 19)


,job_id,job_title,company_size,company_industry,country,remote_type,experience_level,years_experience,education_level,skills_python,skills_sql,skills_ml,skills_deep_learning,skills_cloud,salary,job_posting_month,job_posting_year,hiring_urgency,job_openings
0,1380,Machine Learning Engineer,Enterprise,Technology,Canada,Hybrid,Entry,4,Master,0,1,1,1,1,120000,3,2026,High,2
1,11591,Data Scientist,Startup,Education,Australia,Remote,Entry,4,Master,1,1,0,1,0,70000,12,2022,High,1
2,6145,Data Analyst,Startup,Finance,UK,Onsite,Entry,4,Bachelor,1,1,1,1,1,70000,3,2025,Medium,2
3,1273,Data Engineer,Medium,Education,Australia,Onsite,Entry,4,Master,1,0,1,1,1,71794,12,2025,Medium,1
4,6315,Data Scientist,Startup,E-commerce,India,Remote,Entry,4,Bachelor,1,0,0,1,1,70000,12,2021,Low,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16883,5138,Data Analyst,MNC,Technology,USA,Remote,Mid,5,PhD,0,1,1,1,1,71177,6,2023,High,2
16884,13541,Data Engineer,Startup,Healthcare,Germany,Remote,Mid,5,PhD,0,0,1,1,1,120000,4,2023,High,3
16885,4105,Machine Learning Engineer,Medium,Retail,India,Remote,Mid,5,PhD,1,1,1,0,1,115847,1,2023,Low,4
16886,6516,Data Scientist,Medium,Technology,Australia,Remote,Mid,5,PhD,0,1,1,1,1,91028,2,2025,High,6


In [6]:
dtypes_df = pd.DataFrame(ai_df.dtypes, columns=['Dtype'])
null_df = pd.DataFrame(ai_df.isnull().sum(), columns=['Null Count'])
nunique_df = pd.DataFrame(ai_df.nunique(), columns=['Unique Count'])
decription_df = pd.DataFrame(ai_df.describe())

In [7]:
dtypes_df

,Dtype
job_id,int64
job_title,object
company_size,object
company_industry,object
country,object
remote_type,object
experience_level,object
years_experience,int64
education_level,object
skills_python,int64


In [8]:
null_df

,Null Count
job_id,0
job_title,0
company_size,0
company_industry,0
country,0
remote_type,0
experience_level,0
years_experience,0
education_level,0
skills_python,0


In [9]:
nunique_df

,Unique Count
job_id,10547
job_title,6
company_size,4
company_industry,6
country,7
remote_type,3
experience_level,3
years_experience,9
education_level,3
skills_python,2


In [10]:
decription_df

,job_id,years_experience,skills_python,skills_sql,skills_ml,skills_deep_learning,skills_cloud,salary,job_posting_month,job_posting_year,job_openings
count,16888.000000,16888.000000,16888.000000,16888.000000,16888.000000,16888.000000,16888.000000,16888.000000,16888.000000,16888.000000,16888.000000
mean,5430.909225,5.479157,0.782804,0.787956,0.796897,0.793226,0.792752,124033.212340,6.139152,2022.936878,2.489756
std,4073.909213,4.630177,0.412349,0.408768,0.402320,0.405004,0.405347,37234.368156,3.936000,1.848446,1.497481
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,70000.000000,1.000000,2020.000000,1.000000
25%,2026.750000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,96966.500000,2.000000,2021.000000,1.000000
50%,5138.000000,5.000000,1.000000,1.000000,1.000000,1.000000,1.000000,120000.000000,6.000000,2023.000000,2.000000
75%,8210.250000,7.000000,1.000000,1.000000,1.000000,1.000000,1.000000,144643.250000,10.000000,2024.000000,3.000000
max,26496.000000,15.000000,1.000000,1.000000,1.000000,1.000000,1.000000,300000.000000,12.000000,2026.000000,8.000000


---

## EXPLORATORY DATA ANALYSIS

In [11]:
### 1. Salary Distribution Overview
fig = px.histogram(ai_df, x='salary', nbins=50, title='Overall Salary Distribution',
                   labels={'salary': 'Salary (USD)', 'count': 'Number of Postings'},
                   color_discrete_sequence=['#636EFA'])
fig.update_layout(bargap=0.05)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_01.png', scale=2)
fig.show()

In [12]:
### 2. Salary by Experience Level
fig = px.box(ai_df, x='experience_level', y='salary', color='experience_level',
             title='Salary Distribution by Experience Level',
             category_orders={'experience_level': ['Entry', 'Mid', 'Senior']},
             labels={'salary': 'Salary (USD)', 'experience_level': 'Experience Level'})

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_02.png', scale=2)
fig.show()

In [13]:
### 3. Salary vs. Years of Experience (scatter with trendline)
fig = px.scatter(ai_df, x='years_experience', y='salary', color='experience_level',
                 trendline='lowess', title='Salary vs. Years of Experience',
                 category_orders={'experience_level': ['Entry', 'Mid', 'Senior']},
                 labels={'years_experience': 'Years of Experience', 'salary': 'Salary (USD)'},
                 opacity=0.3)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_03.png', scale=2)
fig.show()

In [14]:
### 4. Average Salary Over Time (by year)
salary_by_year = ai_df.groupby('job_posting_year')['salary'].mean().reset_index()
fig = px.line(salary_by_year, x='job_posting_year', y='salary',
              title='Average Salary Trend Over Time (2020–2026)',
              labels={'job_posting_year': 'Year', 'salary': 'Average Salary (USD)'},
              markers=True)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_04.png', scale=2)
fig.show()

In [15]:
### 5. Salary by Remote Type
fig = px.box(ai_df, x='remote_type', y='salary', color='remote_type',
             title='Salary Distribution: Remote vs. Hybrid vs. Onsite',
             labels={'remote_type': 'Work Type', 'salary': 'Salary (USD)'})

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_05.png', scale=2)
fig.show()

In [16]:
### 6. Salary by Country
country_order = ai_df.groupby('country')['salary'].median().sort_values(ascending=False).index.tolist()
fig = px.box(ai_df, x='country', y='salary', color='country',
             title='Salary Distribution by Country',
             category_orders={'country': country_order},
             labels={'country': 'Country', 'salary': 'Salary (USD)'})

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_06.png', scale=2)
fig.show()

In [17]:
### 7. Salary Heatmap: Company Size × Industry
pivot = ai_df.pivot_table(values='salary', index='company_industry', columns='company_size', aggfunc='mean')
if hasattr(pivot, 'to_pandas'):
    pivot = pivot.to_pandas()
fig = px.imshow(pivot, text_auto='.0f', aspect='auto',
                title='Average Salary by Company Size and Industry',
                labels={'x': 'Company Size', 'y': 'Industry', 'color': 'Avg Salary (USD)'},
                color_continuous_scale='Viridis')

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_07.png', scale=2)
fig.show()

In [18]:
### 8. Average Salary by Experience Level Over Time
salary_exp_year = ai_df.groupby(['job_posting_year', 'experience_level'])['salary'].mean().reset_index()
fig = px.line(salary_exp_year, x='job_posting_year', y='salary', color='experience_level',
              title='Average Salary by Experience Level Over Time',
              category_orders={'experience_level': ['Entry', 'Mid', 'Senior']},
              labels={'job_posting_year': 'Year', 'salary': 'Average Salary (USD)'},
              markers=True)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_08.png', scale=2)
fig.show()

In [19]:
### 9. Experience Level Distribution Over Time
exp_year_counts = ai_df.groupby(['job_posting_year', 'experience_level']).size().reset_index(name='count')
exp_year_pct = exp_year_counts.copy()
totals = exp_year_pct.groupby('job_posting_year')['count'].transform('sum')
exp_year_pct['pct'] = exp_year_pct['count'] / totals * 100

fig = px.bar(exp_year_pct, x='job_posting_year', y='pct', color='experience_level',
             title='Experience Level Demand Over Time (% of Postings)',
             category_orders={'experience_level': ['Entry', 'Mid', 'Senior']},
             labels={'job_posting_year': 'Year', 'pct': '% of Postings'},
             barmode='stack')
fig.add_vline(x=2024.5, line_dash='dash', line_color='red',
              annotation_text='Rise of "Vibe Coding" (2025)', annotation_position='top left')

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_09.png', scale=2)
fig.show()

In [20]:
### 10. Job Postings Count by Year
postings_year = ai_df.groupby('job_posting_year').size().reset_index(name='count')
fig = px.bar(postings_year, x='job_posting_year', y='count',
             title='Total Job Postings per Year',
             labels={'job_posting_year': 'Year', 'count': 'Number of Postings'},
             text_auto=True)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_10.png', scale=2)
fig.show()

In [21]:
### 11. Hiring by Country Over Time
country_year = ai_df.groupby(['job_posting_year', 'country']).size().reset_index(name='count')
fig = px.area(country_year, x='job_posting_year', y='count', color='country',
              title='Job Postings by Country Over Time',
              labels={'job_posting_year': 'Year', 'count': 'Number of Postings'})

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_11.png', scale=2)
fig.show()

In [22]:
### 12. Remote Type Trends Over Time
remote_year = ai_df.groupby(['job_posting_year', 'remote_type']).size().reset_index(name='count')
remote_year_pct = remote_year.copy()
totals = remote_year_pct.groupby('job_posting_year')['count'].transform('sum')
remote_year_pct['pct'] = remote_year_pct['count'] / totals * 100

fig = px.line(remote_year_pct, x='job_posting_year', y='pct', color='remote_type',
              title='Remote vs. Hybrid vs. Onsite Trends Over Time (% of Postings)',
              labels={'job_posting_year': 'Year', 'pct': '% of Postings', 'remote_type': 'Work Type'},
              markers=True)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_12.png', scale=2)
fig.show()

In [23]:
### 13. Skills Required Over Time
skill_cols = ['skills_python', 'skills_sql', 'skills_ml', 'skills_deep_learning', 'skills_cloud']
skills_by_year = ai_df.groupby('job_posting_year')[skill_cols].mean().reset_index()
skills_melted = skills_by_year.melt(id_vars='job_posting_year', var_name='skill', value_name='proportion')
skills_melted['skill'] = skills_melted['skill'].str.replace('skills_', '').str.replace('_', ' ').str.title()

fig = px.line(skills_melted, x='job_posting_year', y='proportion', color='skill',
              title='Skills Demand Over Time (Proportion of Postings Requiring Each Skill)',
              labels={'job_posting_year': 'Year', 'proportion': 'Proportion of Postings', 'skill': 'Skill'},
              markers=True)
fig.update_layout(yaxis_tickformat='.0%')

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_13.png', scale=2)
fig.show()

In [24]:
### 14. Skills by Experience Level
skills_by_exp = ai_df.groupby('experience_level')[skill_cols].mean().reindex(['Entry', 'Mid', 'Senior'])
skills_exp_melted = skills_by_exp.reset_index().melt(id_vars='experience_level', var_name='skill', value_name='proportion')
skills_exp_melted['skill'] = skills_exp_melted['skill'].str.replace('skills_', '').str.replace('_', ' ').str.title()

fig = px.bar(skills_exp_melted, x='experience_level', y='proportion', color='skill',
             title='Skills Required by Experience Level',
             labels={'experience_level': 'Experience Level', 'proportion': 'Proportion Requiring Skill'},
             barmode='group')
fig.update_layout(yaxis_tickformat='.0%')

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_14.png', scale=2)
fig.show()

In [25]:
### 15. Top Skill Combinations
ai_df['skill_combo'] = (
    ai_df[skill_cols].apply(
        lambda row: ' + '.join([c.replace('skills_', '').replace('_', ' ').title()
                                for c in skill_cols if row[c] == 1]) or 'None',
        axis=1
    )
)
combo_counts = ai_df['skill_combo'].value_counts().head(15).reset_index()
combo_counts.columns = ['Skill Combination', 'Count']
fig = px.bar(combo_counts, x='Count', y='Skill Combination', orientation='h',
             title='Top 15 Skill Combinations in Job Postings',
             labels={'Count': 'Number of Postings'})
fig.update_layout(yaxis={'categoryorder': 'total ascending'})

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_15.png', scale=2)
fig.show()

In [26]:
### 16. Postings per Month — Seasonal Trends
monthly = ai_df.groupby('job_posting_month').size().reset_index(name='count')
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly['month_name'] = monthly['job_posting_month'].map(lambda m: month_names[m - 1])

fig = px.bar(monthly, x='month_name', y='count',
             title='Job Postings by Month (All Years Combined)',
             labels={'month_name': 'Month', 'count': 'Number of Postings'},
             text_auto=True)
fig.update_layout(xaxis={'categoryorder': 'array', 'categoryarray': month_names})

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_16.png', scale=2)
fig.show()

In [27]:
### 17. Entry/Junior Level Postings by Month — When do companies hire entry level?
entry_monthly = ai_df[ai_df['experience_level'] == 'Entry'].groupby('job_posting_month').size().reset_index(name='count')
entry_monthly['month_name'] = entry_monthly['job_posting_month'].map(lambda m: month_names[m - 1])

fig = px.bar(entry_monthly, x='month_name', y='count',
             title='Entry-Level Postings by Month (When Are Companies Hiring Junior Devs?)',
             labels={'month_name': 'Month', 'count': 'Entry-Level Postings'},
             text_auto=True, color_discrete_sequence=['#EF553B'])
fig.update_layout(xaxis={'categoryorder': 'array', 'categoryarray': month_names})

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_17.png', scale=2)
fig.show()

In [28]:
### 18. Job Title Distribution and Salary Comparison
fig = px.violin(ai_df, x='job_title', y='salary', color='job_title', box=True,
                title='Salary Distribution by Job Title',
                labels={'job_title': 'Job Title', 'salary': 'Salary (USD)'})
fig.update_layout(showlegend=False)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_18.png', scale=2)
fig.show()

In [29]:
### 19. Industry Hiring Trends Over Time
industry_year = ai_df.groupby(['job_posting_year', 'company_industry']).size().reset_index(name='count')
fig = px.line(industry_year, x='job_posting_year', y='count', color='company_industry',
              title='Hiring Trends by Industry Over Time',
              labels={'job_posting_year': 'Year', 'count': 'Number of Postings', 'company_industry': 'Industry'},
              markers=True)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_19.png', scale=2)
fig.show()

In [30]:
### 20. Salary by Education Level
fig = px.box(ai_df, x='education_level', y='salary', color='education_level',
             title='Salary Distribution by Education Level',
             category_orders={'education_level': ['Bachelor', 'Master', 'PhD']},
             labels={'education_level': 'Education Level', 'salary': 'Salary (USD)'})

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_20.png', scale=2)
fig.show()

In [31]:
### 21. Hiring Urgency Trends Over Time
urgency_year = ai_df.groupby(['job_posting_year', 'hiring_urgency']).size().reset_index(name='count')
urgency_pct = urgency_year.copy()
totals = urgency_pct.groupby('job_posting_year')['count'].transform('sum')
urgency_pct['pct'] = urgency_pct['count'] / totals * 100

fig = px.bar(urgency_pct, x='job_posting_year', y='pct', color='hiring_urgency',
             title='Hiring Urgency Trends Over Time',
             category_orders={'hiring_urgency': ['Low', 'Medium', 'High']},
             labels={'job_posting_year': 'Year', 'pct': '% of Postings'},
             barmode='stack')

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_21.png', scale=2)
fig.show()

In [32]:
### 22. Correlation Heatmap of Numerical Features
numeric_cols = ['years_experience', 'skills_python', 'skills_sql', 'skills_ml',
                'skills_deep_learning', 'skills_cloud', 'salary', 'job_posting_year', 'job_openings']
corr = ai_df[numeric_cols].corr()
if hasattr(corr, 'to_pandas'):
    corr = corr.to_pandas()
fig = px.imshow(corr, text_auto='.2f', aspect='auto',
                title='Correlation Matrix of Numerical Features',
                color_continuous_scale='RdBu_r', zmin=-1, zmax=1)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_22.png', scale=2)
fig.show()

In [33]:
### 23. Company Size Distribution Over Time
size_year = ai_df.groupby(['job_posting_year', 'company_size']).size().reset_index(name='count')
fig = px.bar(size_year, x='job_posting_year', y='count', color='company_size',
             title='Company Size Distribution of Postings Over Time',
             labels={'job_posting_year': 'Year', 'count': 'Number of Postings', 'company_size': 'Company Size'},
             barmode='group')

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_23.png', scale=2)
fig.show()

In [34]:
### 24. Education Level vs. Experience Level — Salary Heatmap
edu_exp_salary = ai_df.pivot_table(values='salary', index='education_level', columns='experience_level', aggfunc='mean')
if hasattr(edu_exp_salary, 'to_pandas'):
    edu_exp_salary = edu_exp_salary.to_pandas()
edu_exp_salary = edu_exp_salary.reindex(index=['Bachelor', 'Master', 'PhD'], columns=['Entry', 'Mid', 'Senior'])
fig = px.imshow(edu_exp_salary, text_auto='.0f', aspect='auto',
                title='Average Salary: Education Level × Experience Level',
                labels={'x': 'Experience Level', 'y': 'Education Level', 'color': 'Avg Salary (USD)'},
                color_continuous_scale='YlOrRd')

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_24.png', scale=2)
fig.show()

In [35]:
### 25. Job Openings by Experience Level Over Time
openings_exp = ai_df.groupby(['job_posting_year', 'experience_level'])['job_openings'].sum().reset_index()
fig = px.bar(openings_exp, x='job_posting_year', y='job_openings', color='experience_level',
             title='Total Job Openings by Experience Level Over Time',
             category_orders={'experience_level': ['Entry', 'Mid', 'Senior']},
             labels={'job_posting_year': 'Year', 'job_openings': 'Total Openings'},
             barmode='group')

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_25.png', scale=2)
fig.show()

In [36]:
### 26. Salary by Remote Type Over Time
salary_remote_year = ai_df.groupby(['job_posting_year', 'remote_type'])['salary'].mean().reset_index()
fig = px.line(salary_remote_year, x='job_posting_year', y='salary', color='remote_type',
              title='Average Salary by Work Type Over Time',
              labels={'job_posting_year': 'Year', 'salary': 'Average Salary (USD)', 'remote_type': 'Work Type'},
              markers=True)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_26.png', scale=2)
fig.show()

In [37]:
### 27. Entry-Level Salary by Remote Type
entry_df = ai_df[ai_df['experience_level'] == 'Entry']
fig = px.box(entry_df, x='remote_type', y='salary', color='remote_type',
             title='Entry-Level Salary Distribution: Remote vs. Hybrid vs. Onsite',
             labels={'remote_type': 'Work Type', 'salary': 'Salary (USD)'})

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_27.png', scale=2)
fig.show()

In [38]:
### 28. Entry-Level Salary by Industry
entry_industry = entry_df.groupby('company_industry')['salary'].median().sort_values(ascending=False).index.tolist()
fig = px.box(entry_df, x='company_industry', y='salary', color='company_industry',
             title='Entry-Level Salary Distribution by Industry',
             category_orders={'company_industry': entry_industry},
             labels={'company_industry': 'Industry', 'salary': 'Salary (USD)'})
fig.update_layout(showlegend=False)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_28.png', scale=2)
fig.show()

In [39]:
### 29. Entry-Level Skill Requirements Over Time
entry_skills_year = entry_df.groupby('job_posting_year')[skill_cols].mean().reset_index()
entry_skills_melted = entry_skills_year.melt(id_vars='job_posting_year', var_name='skill', value_name='proportion')
entry_skills_melted['skill'] = entry_skills_melted['skill'].str.replace('skills_', '').str.replace('_', ' ').str.title()

fig = px.line(entry_skills_melted, x='job_posting_year', y='proportion', color='skill',
              title='Entry-Level Skills Demand Over Time (Proportion of Entry Postings Requiring Each Skill)',
              labels={'job_posting_year': 'Year', 'proportion': 'Proportion of Postings', 'skill': 'Skill'},
              markers=True)
fig.update_layout(yaxis_tickformat='.0%')

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_29.png', scale=2)
fig.show()

In [40]:
### 30. Entry-Level Share of Postings Over Time
exp_counts = ai_df.groupby(['job_posting_year', 'experience_level']).size().reset_index(name='count')
totals_by_year = exp_counts.groupby('job_posting_year')['count'].transform('sum')
exp_counts['share'] = exp_counts['count'] / totals_by_year
entry_share = exp_counts[exp_counts['experience_level'] == 'Entry'].copy()

fig = px.line(entry_share, x='job_posting_year', y='share',
              title='Entry-Level Share of All Job Postings Over Time',
              labels={'job_posting_year': 'Year', 'share': 'Entry-Level Share'},
              markers=True)
fig.update_layout(yaxis_tickformat='.1%')
fig.update_traces(line=dict(color='#EF553B', width=3))

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_30.png', scale=2)
fig.show()

In [41]:
### 31. Entry-Level Salary Over Time
entry_df = ai_df[ai_df['experience_level'] == 'Entry'].groupby('job_posting_year')['salary'].mean().reset_index()

fig = px.line(entry_df, x='job_posting_year', y='salary',
              title='Entry-Level Salary Over Time',
              labels={'job_posting_year': 'Year', 'salary': 'Average Salary (USD)'},
              markers=True)

fig.update_traces(line=dict(color='#64AD9A', width=3))

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/eda_31.png', scale=2)
fig.show()

---

## MACHINE LEARNING

In [4]:
### ML Feature Preparation (shared across models)

# Encode categorical columns
ml_df = ai_df.copy()
label_encoders = {}
cat_cols = ['job_title', 'company_size', 'company_industry', 'country',
            'remote_type', 'experience_level', 'education_level', 'hiring_urgency']
for col in cat_cols:
    le = LabelEncoder()
    col_values = ml_df[col]
    if hasattr(col_values, 'to_pandas'):
        col_values = col_values.to_pandas()
    ml_df[col] = le.fit_transform(col_values)
    label_encoders[col] = le

feature_cols = ['job_title', 'company_size', 'company_industry', 'country', 'remote_type',
                'experience_level', 'years_experience', 'education_level',
                'skills_python', 'skills_sql', 'skills_ml', 'skills_deep_learning',
                'skills_cloud', 'job_posting_month', 'job_posting_year', 'job_openings']

print("Feature columns prepared. Label encoders stored.")
print(f"Shape: {ml_df.shape}")

Feature columns prepared. Label encoders stored.
Shape: (16888, 19)


In [52]:
### ML 1: Salary Prediction — Random Forest vs. Gradient Boosting vs. Linear Regression

# Prepare data
X = ml_df[feature_cols].astype('float32')
y = ml_df['salary'].astype('float32')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def to_cpu(x):
    if hasattr(x, 'to_numpy'):
        return x.to_numpy()
    return x

models = {
    'Linear Regression (cuML)': cuLinearRegression(),
    'Random Forest (cuML)': cuRFRegressor(n_estimators=333, random_state=42),
    'Gradient Boosting (sklearn)': GradientBoostingRegressor(n_estimators=333, random_state=42),
}

results = []
for name, model in models.items():
    if 'sklearn' in name:
        model.fit(to_cpu(X_train), to_cpu(y_train))
        preds = model.predict(to_cpu(X_test))
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    y_test_cpu = to_cpu(y_test)
    preds_cpu = to_cpu(preds)

    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test_cpu, preds_cpu),
        'RMSE': mean_squared_error(y_test_cpu, preds_cpu) ** 0.5,
        'R²': r2_score(y_test_cpu, preds_cpu),
    })

results_df = pd.DataFrame(results)
print("=== Salary Prediction Model Comparison ===")
print(results_df.to_string(index=False))

# Feature importance (using cuRFRegressor for visualization)
gbr_model = GradientBoostingRegressor(n_estimators=333, random_state=42)
gbr_model.fit(to_cpu(X_train), to_cpu(y_train))

importance = pd.DataFrame({'Feature': feature_cols, 'Importance': gbr_model.feature_importances_})
importance = importance.sort_values('Importance', ascending=False)
fig = px.bar(importance, x='Importance', y='Feature', orientation='h',
             title='Feature Importance for Salary Prediction (Random Forest)',
             labels={'Importance': 'Importance Score'})
fig.update_layout(yaxis={'categoryorder': 'total ascending'})

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/ml01_01.png', scale=2)
fig.show()

=== Salary Prediction Model Comparison ===
                      Model          MAE         RMSE       R²
   Linear Regression (cuML) 26500.003906 33919.047156 0.182302
       Random Forest (cuML) 22002.154297 30292.415949 0.347811
Gradient Boosting (sklearn) 21259.041071 29649.772159 0.375189


In [57]:
### ML 2: Experience Level Classification

X = ml_df[['job_title', 'company_size', 'company_industry', 'country', 'remote_type',
           'years_experience', 'education_level', 'skills_python', 'skills_sql',
           'skills_ml', 'skills_deep_learning', 'skills_cloud', 'salary',
           'job_posting_year', 'job_openings']].astype('float32')
y = ml_df['experience_level'].astype('int32')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.17, random_state=42)

rf_clf = cuRFClassifier(n_estimators=333, n_bins=200, n_streams=8, random_state=42)
rf_clf.fit(X_train, y_train)
preds = rf_clf.predict(X_test)

y_test_cpu = to_cpu(y_test)
preds_cpu = to_cpu(preds)

print("=== Experience Level Classification (Random Forest - cuML) ===")
print(f"Accuracy: {accuracy_score(y_test_cpu, preds_cpu):.4f}\n")
target_names = label_encoders['experience_level'].classes_
print(classification_report(y_test_cpu, preds_cpu, target_names=target_names))

# Confusion matrix
cm = confusion_matrix(y_test_cpu, preds_cpu)
fig = px.imshow(cm, text_auto=True, x=target_names, y=target_names,
                title='Confusion Matrix — Experience Level Classification',
                labels={'x': 'Predicted', 'y': 'Actual', 'color': 'Count'},
                color_continuous_scale='Blues')

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/ml02_01.png', scale=2)
fig.show()

=== Experience Level Classification (Random Forest - cuML) ===
Accuracy: 0.6555

              precision    recall  f1-score   support

       Entry       0.71      0.95      0.81       526
         Mid       0.61      0.45      0.52      1096
      Senior       0.65      0.71      0.68      1249

    accuracy                           0.66      2871
   macro avg       0.66      0.70      0.67      2871
weighted avg       0.65      0.66      0.64      2871



In [58]:
### ML 3: Hiring Urgency Prediction
X = ml_df[feature_cols].astype('float32')
y = ml_df['hiring_urgency'].astype('int32')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.17, random_state=42)

rf_urgency = cuRFClassifier(n_estimators=333, n_bins=200, n_streams=8, random_state=42)
rf_urgency.fit(X_train, y_train)
preds = rf_urgency.predict(X_test)

y_test_cpu = to_cpu(y_test)
preds_cpu = to_cpu(preds)

print("=== Hiring Urgency Prediction (Random Forest - cuML) ===")
print(f"Accuracy: {accuracy_score(y_test_cpu, preds_cpu):.4f}\n")
target_names_urgency = label_encoders['hiring_urgency'].classes_
print(classification_report(y_test_cpu, preds_cpu, target_names=target_names_urgency))

cm = confusion_matrix(y_test_cpu, preds_cpu)
fig = px.imshow(cm, text_auto=True, x=target_names_urgency, y=target_names_urgency,
                title='Confusion Matrix — Hiring Urgency Prediction',
                labels={'x': 'Predicted', 'y': 'Actual', 'color': 'Count'},
                color_continuous_scale='Oranges')

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/ml03_01.png', scale=2)
fig.show()

=== Hiring Urgency Prediction (Random Forest - cuML) ===
Accuracy: 0.5681

              precision    recall  f1-score   support

        High       0.59      0.93      0.72      1569
         Low       0.48      0.17      0.25       695
      Medium       0.36      0.09      0.14       607

    accuracy                           0.57      2871
   macro avg       0.48      0.40      0.37      2871
weighted avg       0.51      0.57      0.48      2871



In [60]:
### ML 4: Remote Type Prediction
X = ml_df[['job_title', 'company_size', 'company_industry', 'country',
           'experience_level', 'years_experience', 'education_level',
           'skills_python', 'skills_sql', 'skills_ml', 'skills_deep_learning',
           'skills_cloud', 'salary', 'job_posting_year', 'job_openings', 'hiring_urgency']].astype('float32')
y = ml_df['remote_type'].astype('int32')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.17, random_state=42)

rf_remote = cuRFClassifier(n_estimators=333, n_bins=200, n_streams=8, random_state=42)
rf_remote.fit(X_train, y_train)
preds = rf_remote.predict(X_test)

y_test_cpu = to_cpu(y_test)
preds_cpu = to_cpu(preds)

print("=== Remote Type Prediction (Random Forest - cuML) ===")
print(f"Accuracy: {accuracy_score(y_test_cpu, preds_cpu):.4f}\n")
target_names_remote = label_encoders['remote_type'].classes_
print(classification_report(y_test_cpu, preds_cpu, target_names=target_names_remote))

cm = confusion_matrix(y_test_cpu, preds_cpu)
fig = px.imshow(cm, text_auto=True, x=target_names_remote, y=target_names_remote,
                title='Confusion Matrix — Remote Type Prediction',
                labels={'x': 'Predicted', 'y': 'Actual', 'color': 'Count'},
                color_continuous_scale='Greens')

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/ml04_01.png', scale=2)
fig.show()

=== Remote Type Prediction (Random Forest - cuML) ===
Accuracy: 0.4862

              precision    recall  f1-score   support

      Hybrid       0.42      0.54      0.47       901
      Onsite       0.40      0.23      0.29       781
      Remote       0.58      0.61      0.60      1189

    accuracy                           0.49      2871
   macro avg       0.47      0.46      0.45      2871
weighted avg       0.48      0.49      0.47      2871



In [61]:
### ML 5: Job Market Growth Analysis — Trend and Year-over-Year Growth
# 2026 only contains 1 quarter of data (Jan-Mar). 
# To compute meaningful year-over-year growth and linear trends,
# we annualize the 2026 data by multiplying it by 4 (12 months / 3 months).

# --- 1. Annual posting counts and YoY growth ---
annual_counts = ai_df.groupby('job_posting_year').size().reset_index(name='postings')

# Annualize 2026
months_in_2026 = ai_df[ai_df['job_posting_year'] == 2026]['job_posting_month'].nunique()
annualize_factor = 12 / months_in_2026 if months_in_2026 > 0 else 1

annual_counts['annualized_postings'] = annual_counts['postings'].copy()
annual_counts.loc[annual_counts['job_posting_year'] == 2026, 'annualized_postings'] *= annualize_factor
annual_counts['yoy_growth'] = annual_counts['annualized_postings'].pct_change() * 100

# --- 2. Entry-level annual counts and YoY growth ---
entry_annual = ai_df[ai_df['experience_level'] == 'Entry'].groupby('job_posting_year').size().reset_index(name='entry_postings')
annual_counts = annual_counts.merge(entry_annual, on='job_posting_year')

annual_counts['annualized_entry_postings'] = annual_counts['entry_postings'].copy()
annual_counts.loc[annual_counts['job_posting_year'] == 2026, 'annualized_entry_postings'] *= annualize_factor
annual_counts['entry_yoy_growth'] = annual_counts['annualized_entry_postings'].pct_change() * 100

# --- 3. Linear trend fit (using annualized data) ---
years = annual_counts['job_posting_year'].values.astype(float)
counts_annualized = annual_counts['annualized_postings'].values.astype(float)
slope, intercept = np.polyfit(years, counts_annualized, 1)
trend_line = slope * years + intercept

print('=== Job Market Growth Analysis ===')
print(f'Linear trend: ~{slope:.0f} additional postings per year (based on annualized data)')
print(f'\nYear-over-Year Growth Rates:')
for _, row in annual_counts.iterrows():
    year = int(row['job_posting_year'])
    yoy = f"{row['yoy_growth']:+.1f}%" if not np.isnan(row['yoy_growth']) else 'N/A'
    entry_yoy = f"{row['entry_yoy_growth']:+.1f}%" if not np.isnan(row['entry_yoy_growth']) else 'N/A'
    
    if year == 2026:
        print(f"  {year}: {int(row['postings']):,} actual -> {int(row['annualized_postings']):,} annualized ({yoy})  |  {int(row['entry_postings']):,} actual -> {int(row['annualized_entry_postings']):,} entry annualized ({entry_yoy})")
    else:
        print(f"  {year}: {int(row['postings']):,} total ({yoy})  |  {int(row['entry_postings']):,} entry-level ({entry_yoy})")

# --- 4. Visualization: Total vs Entry postings with trend ---
fig = make_subplots(rows=1, cols=2, subplot_titles=('Total Job Postings with Linear Trend',
                                                      'Year-over-Year Growth: Total vs. Entry-Level'))

# Subplot 1: Total Postings (use annualized for all so it's a single bar series, but color 2026 differently)
colors = ['#636EFA'] * len(annual_counts)
colors[-1] = 'rgba(99, 110, 250, 0.5)' # Lighter color for projected 2026

fig.add_trace(go.Bar(x=annual_counts['job_posting_year'], 
                     y=annual_counts['annualized_postings'],
                     name='Annualized Postings', marker_color=colors,
                     text=annual_counts['annualized_postings'].apply(lambda x: f"{x:,.0f}{'*' if x == annual_counts['annualized_postings'].iloc[-1] else ''}"),
                     textposition='auto'), row=1, col=1)

fig.add_trace(go.Scatter(x=annual_counts['job_posting_year'], y=trend_line,
                         mode='lines', name=f'Trend (+{slope:.0f}/yr)',
                         line=dict(dash='dash', color='red', width=2)), row=1, col=1)

# Subplot 2: YoY Growth
fig.add_trace(go.Bar(x=annual_counts['job_posting_year'], y=annual_counts['yoy_growth'],
                     name='Total YoY %', marker_color='#636EFA'), row=1, col=2)
fig.add_trace(go.Bar(x=annual_counts['job_posting_year'], y=annual_counts['entry_yoy_growth'],
                     name='Entry-Level YoY %', marker_color='#EF553B'), row=1, col=2)

fig.update_layout(title='Job Market Growth Analysis: Overall Trend and Entry-Level Comparison<br><sup>*2026 figures are annualized based on Q1 data</sup>',
                  height=800, barmode='group')
fig.update_xaxes(title_text='Year', row=1, col=1)
fig.update_xaxes(title_text='Year', row=1, col=2)
fig.update_yaxes(title_text='Number of Postings', row=1, col=1)
fig.update_yaxes(title_text='Growth Rate (%)', row=1, col=2)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/ml05_01.png', scale=2)
fig.show()

=== Job Market Growth Analysis ===
Linear trend: ~376 additional postings per year (based on annualized data)

Year-over-Year Growth Rates:
  2020: 2,280 total (N/A)  |  389 entry-level (N/A)
  2021: 2,236 total (-1.9%)  |  389 entry-level (+0.0%)
  2022: 2,301 total (+2.9%)  |  405 entry-level (+4.1%)
  2023: 2,962 total (+28.7%)  |  533 entry-level (+31.6%)
  2024: 2,960 total (-0.1%)  |  509 entry-level (-4.5%)
  2025: 2,860 total (-3.4%)  |  487 entry-level (-4.3%)
  2026: 1,289 actual -> 5,156 annualized (+80.3%)  |  209 actual -> 836 entry annualized (+71.7%)


---

## DEEP LEARNING

The sklearn models above tell us *what predicts salary*. Now we ask a different question: **what hidden structure exists in the AI job market?**

Deep neural networks won't beat Gradient Boosting on prediction accuracy for 15k tabular rows — that's well-established. Instead, we use them as **exploration tools**:
- **Entity Embeddings** (TensorFlow/Keras): Learn a continuous "map" of how job categories relate — which roles, countries, and industries the model treats as salary-equivalent
- **Autoencoder Anomaly Detection** (TensorFlow/Keras): Find structurally unusual job postings — unusual *combinations* of features, not just single-column outliers
- **Integrated Gradients** (PyTorch): Per-subgroup feature attribution that shows how salary drivers differ for Entry vs. Senior roles, or across countries

In [5]:
### DNN 1: Entity Embeddings — Learning the Hidden Geometry of Job Categories (TensorFlow/Keras)

# --- Define categorical features with embedding dimensions ---
# Rule of thumb: embedding_dim = min(50, num_unique // 2 + 1)
cat_feature_info = {
    'job_title':        (len(label_encoders['job_title'].classes_), 3),
    'company_size':     (len(label_encoders['company_size'].classes_), 2),
    'company_industry': (len(label_encoders['company_industry'].classes_), 3),
    'country':          (len(label_encoders['country'].classes_), 3),
    'remote_type':      (len(label_encoders['remote_type'].classes_), 2),
    'experience_level': (len(label_encoders['experience_level'].classes_), 2),
    'education_level':  (len(label_encoders['education_level'].classes_), 2),
    'hiring_urgency':   (len(label_encoders['hiring_urgency'].classes_), 2),
}

num_features = ['years_experience', 'skills_python', 'skills_sql', 'skills_ml',
                'skills_deep_learning', 'skills_cloud', 'job_posting_month',
                'job_posting_year', 'job_openings']

# --- Prepare data ---
X_cat = {col: ml_df[col].values for col in cat_feature_info}
X_num = ml_df[num_features].values.astype('float32')
scaler_emb = StandardScaler()
X_num_scaled = scaler_emb.fit_transform(X_num)
y_emb = ml_df['salary'].values.astype('float32')

indices = np.arange(len(y_emb))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

X_cat_train = {col: vals[train_idx] for col, vals in X_cat.items()}
X_cat_test  = {col: vals[test_idx]  for col, vals in X_cat.items()}
X_num_train, X_num_test = X_num_scaled[train_idx], X_num_scaled[test_idx]
y_emb_train, y_emb_test = y_emb[train_idx], y_emb[test_idx]

# --- Build entity embedding model (Keras Functional API) ---
cat_inputs, cat_embeddings = [], []
for col, (n_unique, emb_dim) in cat_feature_info.items():
    inp = keras.Input(shape=(1,), name=f'input_{col}')
    emb = layers.Embedding(input_dim=n_unique, output_dim=emb_dim, name=f'emb_{col}')(inp)
    emb = layers.Flatten()(emb)
    cat_inputs.append(inp)
    cat_embeddings.append(emb)

num_input = keras.Input(shape=(len(num_features),), name='input_numerical')

x = layers.Concatenate()(cat_embeddings + [num_input])

# --- Deep Tabular MLP with Residual Connections ---
# Initial projection to desired width
x = layers.Dense(512, activation='silu')(x)
x = layers.BatchNormalization()(x)

# Residual Block 1
res = x
x = layers.Dense(512, activation='silu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(512)(x)
x = layers.Add()([res, x])
x = layers.Activation('silu')(x)
x = layers.BatchNormalization()(x)

# Residual Block 2 (with dimension reduction)
res = layers.Dense(256)(x) # Match dimensions
x = layers.Dense(256, activation='silu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(256)(x)
x = layers.Add()([res, x])
x = layers.Activation('silu')(x)
x = layers.BatchNormalization()(x)

# Residual Block 3 (with dimension reduction)
res = layers.Dense(128)(x) # Match dimensions
x = layers.Dense(128, activation='silu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)
x = layers.Dense(128)(x)
x = layers.Add()([res, x])
x = layers.Activation('silu')(x)
x = layers.BatchNormalization()(x)

# Residual Block 4 (with dimension reduction)
res = layers.Dense(64)(x) # Match dimensions
x = layers.Dense(64, activation='silu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(64)(x)
x = layers.Add()([res, x])
x = layers.Activation('silu')(x)
x = layers.BatchNormalization()(x)

# Final funnel to output
x = layers.Dense(32, activation='silu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(16, activation='silu')(x)
output = layers.Dense(1, name='salary_output')(x)

emb_model = keras.Model(inputs=cat_inputs + [num_input], outputs=output)
emb_model.compile(optimizer=keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4), loss='mse', metrics=['mae'])
def make_input_dict(X_cat_dict, X_num_array):
    d = {f'input_{col}': vals for col, vals in X_cat_dict.items()}
    d['input_numerical'] = X_num_array
    return d

history_emb = emb_model.fit(
    make_input_dict(X_cat_train, X_num_train), y_emb_train,
    validation_split=0.15, epochs=333, batch_size=256,
    callbacks=[keras.callbacks.EarlyStopping(patience=33, restore_best_weights=True),
               keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)],
    verbose=1
)

# --- Evaluate ---
preds_emb = emb_model.predict(make_input_dict(X_cat_test, X_num_test)).flatten()
dnn_r2 = r2_score(y_emb_test, preds_emb)
dnn_mae = mean_absolute_error(y_emb_test, preds_emb)
print(f"\n=== DNN with Entity Embeddings — Salary Prediction ===")
print(f"MAE:  ${dnn_mae:,.0f}")
print(f"R²:   {dnn_r2:.4f}")
print(f"(Compare to Gradient Boosting R² from ML 1 — GB likely wins on metrics.)")
print(f"(The embeddings below reveal structure GB cannot show.)\n")

# --- Extract embedding weights ---
embedding_data = {}
for col in cat_feature_info:
    emb_layer = emb_model.get_layer(f'emb_{col}')
    weights = emb_layer.get_weights()[0]
    labels = label_encoders[col].classes_
    embedding_data[col] = (labels, weights)

# --- Visualization 1: 2D Embedding Scatter for 3D categories ---
cols_3d = ['job_title', 'company_industry', 'country']
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=[f'{c.replace("_", " ").title()} Embeddings' for c in cols_3d])
colors = px.colors.qualitative.Set2

for i, col in enumerate(cols_3d):
    labels, weights = embedding_data[col]
    for j, label in enumerate(labels):
        fig.add_trace(go.Scatter(
            x=[weights[j, 0]], y=[weights[j, 1]],
            mode='markers+text', text=[label], textposition='top center',
            marker=dict(size=14, color=colors[j % len(colors)]),
            showlegend=False,
        ), row=1, col=i+1)
    fig.update_xaxes(title_text='Embedding Dim 1', row=1, col=i+1)
    fig.update_yaxes(title_text='Embedding Dim 2', row=1, col=i+1)

fig.update_layout(
    title='Learned Entity Embeddings: How the DNN Sees Job Market Categories<br>'
          '<sup>Categories close together have similar salary effects — relationships learned, not hand-coded</sup>',
    height=500, width=1200)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/dl01_01.png', scale=2)
fig.show()

# --- Visualization 2: Cosine distance heatmaps ---
focus_cols = ['job_title', 'country', 'company_industry', 'experience_level']
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f'{c.replace("_", " ").title()}: Embedding Distance' for c in focus_cols])

for idx, col in enumerate(focus_cols):
    labels, weights = embedding_data[col]
    dist_matrix = squareform(pdist(weights, metric='cosine'))
    row, col_idx = divmod(idx, 2)
    fig.add_trace(go.Heatmap(
        z=dist_matrix, x=list(labels), y=list(labels),
        colorscale='Viridis', text=np.round(dist_matrix, 3),
        texttemplate='%{text}', showscale=(idx == 0),
    ), row=row+1, col=col_idx+1)

fig.update_layout(
    title='Embedding Cosine Distance: Which Categories Does the DNN Treat as Equivalent?<br>'
          '<sup>Low distance = model learned these categories have similar salary effects</sup>',
    height=700, width=900)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/dl01_02.png', scale=2)
fig.show()

# --- Training history ---
fig = go.Figure()
fig.add_trace(go.Scatter(y=history_emb.history['loss'], name='Train Loss'))
fig.add_trace(go.Scatter(y=history_emb.history['val_loss'], name='Val Loss'))
fig.update_layout(title='Entity Embedding DNN — Training History',
                  xaxis_title='Epoch', yaxis_title='MSE Loss')

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/dl01_03.png', scale=2)
fig.show()

I0000 00:00:1775644255.665154    5173 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 7358 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:09:00.0, compute capability: 7.5


Epoch 1/333


I0000 00:00:1775644274.692015    5743 service.cc:153] XLA service 0x7f9ae0051a70 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775644274.692060    5743 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 2080 Ti, Compute Capability 7.5 (Driver: 13.2.0; Runtime: 13.0.0; Toolkit: 12.5.0; DNN: 9.13.0)
I0000 00:00:1775644275.112113    5743 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1775644277.359071    5743 cuda_dnn.cc:461] Loaded cuDNN version 91300
I0000 00:00:1775644278.126678    5743 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_12580__.113


16/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 16340330176.0000 - mae: 122713.5571

I0000 00:00:1775644295.695700    5743 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

45/45 ━━━━━━━━━━━━━━━━━━━━ 62s 539ms/step - loss: 16813536256.0000 - mae: 124248.1641 - val_loss: 16625697792.0000 - val_mae: 123352.8359 - learning_rate: 0.0010
Epoch 2/333
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 16812232704.0000 - mae: 124243.0234 - val_loss: 16623609856.0000 - val_mae: 123344.8672 - learning_rate: 0.0010
Epoch 3/333
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 16810085376.0000 - mae: 124234.7969 - val_loss: 16623362048.0000 - val_mae: 123343.8203 - learning_rate: 0.0010
Epoch 4/333
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 16807107584.0000 - mae: 124223.3281 - val_loss: 16618345472.0000 - val_mae: 123325.3125 - learning_rate: 0.0010
Epoch 5/333
45/45 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 16802917376.0000 - mae: 124207.5000 - val_loss: 16608564224.0000 - val_mae: 123287.4609 - learning_rate: 0.0010
Epoch 6/333
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 16797061120.0000 - mae: 124185.7812 - val_loss: 16598190080.0000 - val_mae: 123248.2188 - 

I0000 00:00:1775644415.840386    5743 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_80082__.1


 93/106 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

I0000 00:00:1775644417.952200    5743 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_81876__.1


106/106 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step

=== DNN with Entity Embeddings — Salary Prediction ===
MAE:  $22,000
R²:   0.3372
(Compare to Gradient Boosting R² from ML 1 — GB likely wins on metrics.)
(The embeddings below reveal structure GB cannot show.)



In [50]:
### DNN 2: Autoencoder Anomaly Detection — Finding Structurally Unusual Job Postings (TensorFlow/Keras)

# --- Prepare all features as a single normalized matrix ---
ae_features = feature_cols + ['hiring_urgency', 'salary']
X_ae = ml_df[ae_features].values.astype('float32')
scaler_ae = StandardScaler()
X_ae_scaled = scaler_ae.fit_transform(X_ae)
n_ae_features = X_ae_scaled.shape[1]

# --- Build autoencoder ---
ae_input = keras.Input(shape=(n_ae_features,), name='ae_input')

# --- Encoder ---
x = layers.Dense(256, activation='silu')(ae_input)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.1)(x)

x = layers.Dense(128, activation='silu')(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(64, activation='silu')(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(32, activation='silu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.1)(x)

# --- Bottleneck (Linear activation maximizes representation space) ---
bottleneck = layers.Dense(8, activation='linear', name='bottleneck')(x)

# --- Decoder ---
x = layers.Dense(32, activation='silu')(bottleneck)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.1)(x)

x = layers.Dense(64, activation='silu')(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(128, activation='silu')(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(256, activation='silu')(x)
x = layers.BatchNormalization()(x)

ae_output = layers.Dense(n_ae_features, activation='linear', name='reconstruction')(x)

autoencoder = keras.Model(inputs=ae_input, outputs=ae_output)
autoencoder.compile(optimizer=keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4), loss='mse')
encoder = keras.Model(inputs=ae_input, outputs=autoencoder.get_layer('bottleneck').output, name='encoder')

# --- Train (unsupervised — reconstruct itself) ---
history_ae = autoencoder.fit(
    X_ae_scaled, X_ae_scaled, epochs=333, batch_size=256, validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping(patience=33, restore_best_weights=True),
               keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=13)],
    verbose=1
)

# --- Compute reconstruction error per sample ---
reconstructed = autoencoder.predict(X_ae_scaled)
mse_per_sample = np.mean((X_ae_scaled - reconstructed) ** 2, axis=1)

anomaly_df = ai_df.copy()
anomaly_df['reconstruction_error'] = mse_per_sample
anomaly_df['is_anomaly'] = mse_per_sample > np.percentile(mse_per_sample, 97)

n_anomalies = anomaly_df['is_anomaly'].sum()
print(f"\n=== Autoencoder Anomaly Detection ===")
print(f"Total postings: {len(anomaly_df):,}")
print(f"Anomalies (top 3% reconstruction error): {n_anomalies}")
print(f"Error — Mean: {mse_per_sample.mean():.4f}, 97th pctl: {np.percentile(mse_per_sample, 97):.4f}")

# --- Viz 1: Reconstruction error distribution ---
fig = go.Figure()
fig.add_trace(go.Histogram(x=mse_per_sample[~anomaly_df['is_anomaly'].values],
                           name='Normal', marker_color='steelblue', opacity=0.7))
fig.add_trace(go.Histogram(x=mse_per_sample[anomaly_df['is_anomaly'].values],
                           name='Anomaly (top 3%)', marker_color='crimson', opacity=0.7))
fig.update_layout(
    title='Autoencoder Reconstruction Error Distribution<br>'
          '<sup>High error = unusual feature combination the model cannot compress/reconstruct</sup>',
    xaxis_title='Mean Squared Reconstruction Error', yaxis_title='Count', barmode='overlay')

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/dl02_01.png', scale=2)
fig.show()

# --- Viz 2: Anomaly vs normal category distribution ---
cat_compare = ['job_title', 'company_industry', 'country', 'experience_level',
               'education_level', 'remote_type', 'hiring_urgency', 'company_size']
fig = make_subplots(rows=2, cols=4,
                    subplot_titles=[c.replace('_', ' ').title() for c in cat_compare])

for idx, col in enumerate(cat_compare):
    row, col_idx = divmod(idx, 4)
    normal_pct = anomaly_df[~anomaly_df['is_anomaly']][col].value_counts(normalize=True).sort_index()
    anomaly_pct = anomaly_df[anomaly_df['is_anomaly']][col].value_counts(normalize=True).sort_index()
    all_cats = sorted(set(normal_pct.index) | set(anomaly_pct.index))

    fig.add_trace(go.Bar(x=all_cats, y=[normal_pct.get(c, 0) for c in all_cats],
                         name='Normal', marker_color='steelblue', showlegend=(idx == 0)),
                  row=row+1, col=col_idx+1)
    fig.add_trace(go.Bar(x=all_cats, y=[anomaly_pct.get(c, 0) for c in all_cats],
                         name='Anomaly', marker_color='crimson', showlegend=(idx == 0)),
                  row=row+1, col=col_idx+1)

fig.update_layout(
    title='Anomalous vs. Normal Postings: Category Distribution<br>'
          '<sup>Look for categories over/under-represented in anomalies</sup>',
    height=600, width=1200, barmode='group')

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/dl02_02.png', scale=2)
fig.show()

# --- Viz 3: Salary vs experience colored by anomaly ---
fig = px.scatter(
    anomaly_df, x='years_experience', y='salary', color='is_anomaly',
    color_discrete_map={True: 'crimson', False: 'steelblue'}, opacity=0.4,
    title='Anomalous Job Postings: Salary vs. Experience<br>'
          '<sup>Red = structurally unusual combinations, not just salary outliers</sup>',
    labels={'years_experience': 'Years of Experience', 'salary': 'Salary (USD)', 'is_anomaly': 'Anomaly'},
    hover_data=['job_title', 'company_industry', 'country', 'experience_level'])

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/dl02_03.png', scale=2)
fig.show()

# --- Viz 4: Bottleneck space (PCA projection) ---
bottleneck_repr = encoder.predict(X_ae_scaled)
pca = PCA(n_components=2)
bottleneck_2d = pca.fit_transform(bottleneck_repr)
anomaly_df['bottleneck_x'] = bottleneck_2d[:, 0]
anomaly_df['bottleneck_y'] = bottleneck_2d[:, 1]

fig = px.scatter(
    anomaly_df, x='bottleneck_x', y='bottleneck_y',
    color='experience_level', symbol='is_anomaly',
    symbol_map={True: 'x', False: 'circle'}, opacity=0.5,
    title='Autoencoder Bottleneck Space (PCA Projection)<br>'
          '<sup>Each point = job posting compressed to 8 dims then projected to 2D — X = anomaly</sup>',
    labels={'bottleneck_x': 'PC1', 'bottleneck_y': 'PC2'},
    hover_data=['job_title', 'salary', 'country', 'company_industry'],
    category_orders={'experience_level': ['Entry', 'Mid', 'Senior']})

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/dl02_04.png', scale=2)
fig.show()

# --- Top anomalies ---
print("\n=== Top 10 Most Anomalous Postings ===")
top_anomalies = anomaly_df.nlargest(10, 'reconstruction_error')
display_cols = ['job_title', 'company_size', 'company_industry', 'country', 'remote_type',
                'experience_level', 'years_experience', 'education_level', 'salary',
                'hiring_urgency', 'reconstruction_error']
print(top_anomalies[display_cols].to_string(index=False))

Epoch 1/333


I0000 00:00:1774787993.777713  438241 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_61697__.54


54/55 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.6237

I0000 00:00:1774787999.878331  438243 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_61697__.54


55/55 ━━━━━━━━━━━━━━━━━━━━ 23s 154ms/step - loss: 1.2200 - val_loss: 0.9847 - learning_rate: 0.0010
Epoch 2/333
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.8010 - val_loss: 0.9259 - learning_rate: 0.0010
Epoch 3/333
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.7259 - val_loss: 0.8193 - learning_rate: 0.0010
Epoch 4/333
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.6831 - val_loss: 0.7104 - learning_rate: 0.0010
Epoch 5/333
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6530 - val_loss: 0.6238 - learning_rate: 0.0010
Epoch 6/333
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6297 - val_loss: 0.5769 - learning_rate: 0.0010
Epoch 7/333
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.6080 - val_loss: 0.5465 - learning_rate: 0.0010
Epoch 8/333
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.5888 - val_loss: 0.5250 - learning_rate: 0.0010
Epoch 9/333
55/55 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.5717 - val_loss: 0.5084 - learning_rate: 0.0010
Epoch 10/333
55/55 ━━━━━━━━━

485/485 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step



=== Top 10 Most Anomalous Postings ===
                job_title company_size company_industry   country remote_type experience_level  years_experience education_level  salary hiring_urgency  reconstruction_error
              AI Engineer          MNC       E-commerce Singapore      Remote            Entry                11          Master  152609           High              0.338453
Machine Learning Engineer       Medium          Finance        UK      Onsite              Mid                14             PhD  157508           High              0.316156
              AI Engineer      Startup           Retail    Canada      Remote            Entry                11             PhD  120396           High              0.271362
Machine Learning Engineer       Medium        Education Singapore      Onsite           Senior                12          Master  172230           High              0.252522
           Data Scientist          MNC        Education        UK      Onsite           Se

### DL 3: PyTorch Integrated Gradients — Per-Feature Salary Attribution

**Why PyTorch here?** Random Forest gives one global feature-importance bar chart. Integrated
Gradients (a gradient-based attribution method) gives a *per-sample* importance score for every
feature, letting us ask richer questions: "What drives salary predictions for Senior vs. Entry
roles?" or "How does attribution differ across countries?"

PyTorch's eager-mode autograd makes this cleaner to implement than in TensorFlow. We use
the **Captum** library (Meta's official PyTorch interpretability toolkit). The neural network
itself will not beat Random Forest on 15k tabular rows — that is expected and fine. The point
is the *attribution analysis*, not the prediction accuracy.

In [6]:
# ---------------------------------------------------------------------------
# 1. Prepare data (uses ml_df, feature_cols from the shared ML prep cell)
# ---------------------------------------------------------------------------
X_all = ml_df[feature_cols].values.astype(np.float32)
y_all = ml_df['salary'].values.astype(np.float32)

# Standardize features and target for neural-net training stability
X_mean, X_std = X_all.mean(axis=0), X_all.std(axis=0) + 1e-8
y_mean, y_std = y_all.mean(), y_all.std() + 1e-8
X_scaled = (X_all - X_mean) / X_std
y_scaled = (y_all - y_mean) / y_std

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_train_pt = torch.tensor(X_train_t, device=device)
y_train_pt = torch.tensor(y_train_t, device=device).unsqueeze(1)
X_test_pt  = torch.tensor(X_test_t,  device=device)
y_test_pt  = torch.tensor(y_test_t,  device=device).unsqueeze(1)

# ---------------------------------------------------------------------------
# 2. Define a small MLP — intentionally modest for 15k rows
# ---------------------------------------------------------------------------
class SalaryMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 1024),
            nn.SiLU(),
            nn.BatchNorm1d(1024),
            nn.Dropout(0.2),
            nn.Linear(1024, 512),
            nn.SiLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.SiLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.SiLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.1),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.net(x)

model = SalaryMLP(len(feature_cols)).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
loss_fn = nn.MSELoss()

# ---------------------------------------------------------------------------
# 3. Train (mini-batch, early stopping)
# ---------------------------------------------------------------------------
batch_size = 256
best_val_loss = float('inf')
patience_counter = 0
train_losses, val_losses = [], []

for epoch in range(333):
    model.train()
    perm = torch.randperm(X_train_pt.size(0), device=device)
    epoch_loss = 0.0
    for i in range(0, len(perm), batch_size):
        idx = perm[i:i+batch_size]
        pred = model(X_train_pt[idx])
        loss = loss_fn(pred, y_train_pt[idx])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    train_losses.append(epoch_loss / len(perm))

    model.eval()
    with torch.no_grad():
        val_pred = model(X_test_pt)
        val_loss = loss_fn(val_pred, y_test_pt).item()
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= 25:
            break

model.load_state_dict(best_state)
print(f"Training complete -- stopped at epoch {epoch+1}, best val MSE (scaled): {best_val_loss:.4f}")

# Report R2 in original scale for comparison with sklearn models
model.eval()
with torch.no_grad():
    test_pred_scaled = model(X_test_pt).cpu().numpy().flatten()
test_pred_dollars = test_pred_scaled * y_std + y_mean
y_test_dollars = y_test_t * y_std + y_mean
nn_r2 = r2_score(y_test_dollars, test_pred_dollars)
nn_mae = mean_absolute_error(y_test_dollars, test_pred_dollars)
print(f"PyTorch MLP -- R2: {nn_r2:.4f}, MAE: ${nn_mae:,.0f}")
print("(Expected to underperform tree models on 15k tabular rows -- the value is in attribution below)")

# ---------------------------------------------------------------------------
# 4. Integrated Gradients — implemented from scratch using PyTorch autograd
#    No external library needed. The algorithm:
#    1. Create a straight-line path from baseline (zeros) to input (m steps)
#    2. Compute gradients of the output w.r.t. input at each step
#    3. Attribution = (input - baseline) * mean(gradients along path)
# ---------------------------------------------------------------------------
def integrated_gradients(model, inputs, baseline, n_steps=69):
    """Compute Integrated Gradients attribution for a batch of inputs."""
    model.eval()
    # inputs: (N, D), baseline: (1, D) or (N, D)
    # Generate interpolation alphas: shape (n_steps+1, 1, 1) for broadcasting
    alphas = torch.linspace(0, 1, n_steps + 1, device=inputs.device).view(-1, 1, 1)

    # Interpolated inputs: (n_steps+1, N, D)
    path = baseline.unsqueeze(0) + alphas * (inputs.unsqueeze(0) - baseline.unsqueeze(0))

    # Compute gradients at each interpolation point
    grads = []
    for step in range(n_steps + 1):
        x_step = path[step].detach().requires_grad_(True)
        out = model(x_step).sum()
        out.backward()
        grads.append(x_step.grad.detach())

    # Stack and average using trapezoidal rule
    grads = torch.stack(grads, dim=0)  # (n_steps+1, N, D)
    avg_grads = (grads[:-1] + grads[1:]).mean(dim=0) / 2.0

    # Attribution = (input - baseline) * averaged gradients
    attributions = (inputs - baseline) * avg_grads
    return attributions

baseline = torch.zeros(1, len(feature_cols), device=device)

# Process in batches to avoid OOM on large test sets
attr_chunks = []
chunk_size = 512
for i in range(0, X_test_pt.shape[0], chunk_size):
    chunk = X_test_pt[i:i+chunk_size].detach().clone()
    attr_chunk = integrated_gradients(model, chunk, baseline, n_steps=69)
    attr_chunks.append(attr_chunk.cpu().numpy())

attr_np = np.concatenate(attr_chunks, axis=0)  # shape: (n_test, n_features)

# Sanity check: sum of attributions should approximate (prediction - baseline_prediction)
with torch.no_grad():
    baseline_pred = model(baseline).item()
    test_preds_check = model(X_test_pt).cpu().numpy().flatten()
expected_diff = test_preds_check - baseline_pred
actual_diff = attr_np.sum(axis=1)
completeness_error = np.abs(expected_diff - actual_diff).mean()
print(f"Completeness check (should be small): mean absolute error = {completeness_error:.4f}")

# ---------------------------------------------------------------------------
# 5. Visualization: Global Attribution (mean |attribution| per feature)
# ---------------------------------------------------------------------------
mean_attr = np.abs(attr_np).mean(axis=0)
attr_df = pd.DataFrame({'Feature': feature_cols, 'Mean |Attribution|': mean_attr})
attr_df = attr_df.sort_values('Mean |Attribution|', ascending=False)

fig = px.bar(attr_df, x='Mean |Attribution|', y='Feature', orientation='h',
             title='Integrated Gradients: Global Feature Attribution for Salary (PyTorch MLP)',
             labels={'Mean |Attribution|': 'Mean Absolute Attribution Score'},
             color='Mean |Attribution|', color_continuous_scale='Viridis')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, showlegend=False)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/dl03_01.png', scale=2)
fig.show()

# ---------------------------------------------------------------------------
# 6. Visualization: Attribution by Experience Level
#    "What drives salary differently for Entry vs. Mid vs. Senior?"
# ---------------------------------------------------------------------------
# Recover experience_level for the test split
_, test_indices = train_test_split(range(len(X_all)), test_size=0.2, random_state=42)
exp_levels_test = ml_df.iloc[test_indices]['experience_level'].values
exp_names_test = [label_encoders['experience_level'].classes_[v] for v in exp_levels_test]

# Build attribution-by-group dataframe
rows = []
for level_name in ['Entry', 'Mid', 'Senior']:
    mask = np.array(exp_names_test) == level_name
    if mask.sum() == 0:
        continue
    group_attr = np.abs(attr_np[mask]).mean(axis=0)
    for feat, val in zip(feature_cols, group_attr):
        rows.append({'Experience Level': level_name, 'Feature': feat, 'Attribution': val})

attr_by_exp = pd.DataFrame(rows)

fig = px.bar(attr_by_exp, x='Attribution', y='Feature', color='Experience Level',
             orientation='h', barmode='group',
             title='What Drives Salary Predictions? Attribution by Experience Level',
             category_orders={'Experience Level': ['Entry', 'Mid', 'Senior']},
             labels={'Attribution': 'Mean |Attribution|'})
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/dl03_02.png', scale=2)
fig.show()

# ---------------------------------------------------------------------------
# 7. Visualization: Attribution by Country
#    "Does the model rely on different signals in different markets?"
# ---------------------------------------------------------------------------
countries_test = [label_encoders['country'].classes_[v]
                  for v in ml_df.iloc[test_indices]['country'].values]

rows_c = []
for country_name in label_encoders['country'].classes_:
    mask = np.array(countries_test) == country_name
    if mask.sum() < 20:
        continue
    group_attr = np.abs(attr_np[mask]).mean(axis=0)
    for feat, val in zip(feature_cols, group_attr):
        rows_c.append({'Country': country_name, 'Feature': feat, 'Attribution': val})

attr_by_country = pd.DataFrame(rows_c)

# Show top 6 features per country for readability
top_features = attr_df.head(6)['Feature'].tolist()
attr_by_country_top = attr_by_country[attr_by_country['Feature'].isin(top_features)]

fig = px.bar(attr_by_country_top, x='Attribution', y='Feature', color='Country',
             orientation='h', barmode='group',
             title='Salary Attribution by Country (Top 6 Features)',
             labels={'Attribution': 'Mean |Attribution|'})
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=500)

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig.write_image('notebook_images/plot_images/new/dl03_03.png', scale=2)
fig.show()

print("\n--- Interpretation Guide ---")
print("Higher attribution = that feature contributed MORE to the model's salary prediction.")
print("Unlike Random Forest importance (one global ranking), Integrated Gradients shows")
print("HOW attributions shift across subgroups -- e.g., 'years_experience' may matter more")
print("for Senior roles while 'education_level' matters more for Entry roles.")

Training complete -- stopped at epoch 34, best val MSE (scaled): 0.6716
PyTorch MLP -- R2: 0.3383, MAE: $22,156
(Expected to underperform tree models on 15k tabular rows -- the value is in attribution below)
Completeness check (should be small): mean absolute error = 0.0000



--- Interpretation Guide ---
Higher attribution = that feature contributed MORE to the model's salary prediction.
Unlike Random Forest importance (one global ranking), Integrated Gradients shows
HOW attributions shift across subgroups -- e.g., 'years_experience' may matter more
for Senior roles while 'education_level' matters more for Entry roles.


---
## FORECASTING — Job Market Projections Through 2028

The EDA and ML sections above show what the AI job market looks like *today*. Now we ask: **where is it heading?**

We use a **hybrid forecasting approach**:
1. **SARIMA** on aggregate monthly postings to project total market volume
2. **Proportion models** for each segment (experience level, remote type, industry, country) to forecast *compositional shifts*
3. **Rate models** for skill requirements (Python, SQL, ML, Deep Learning, Cloud)

The key insight: with near-stationary total volume, the interesting story is not "how much" but "what mix" — and that is exactly what matters for someone planning to enter the market.

In [7]:
### Forecast Preparation — Build Monthly Time Series

# --- 1. Aggregate monthly postings ---
monthly_total = (
    ai_df.groupby(['job_posting_year', 'job_posting_month'])
    .size()
    .reset_index(name='count')
)
monthly_total['date'] = pd.to_datetime(
    monthly_total['job_posting_year'].astype(str) + '-'
    + monthly_total['job_posting_month'].astype(str).str.zfill(2) + '-01'
)
monthly_total = monthly_total.sort_values('date').reset_index(drop=True)
monthly_total = monthly_total.set_index('date')

print(f"Monthly total series: {len(monthly_total)} months, "
      f"{monthly_total['count'].min()}-{monthly_total['count'].max()} range")

# --- 2. Segment proportion series ---
segment_dims = {
    'experience_level': ['Entry', 'Mid', 'Senior'],
    'remote_type': ['Remote', 'Hybrid', 'Onsite'],
    'company_industry': ai_df['company_industry'].unique().tolist(),
    'country': ai_df['country'].unique().tolist(),
}

segment_shares = {}  # {dim: DataFrame with date index, one column per category}
for dim, categories in segment_dims.items():
    shares_df = pd.DataFrame(index=monthly_total.index)
    for cat in categories:
        cat_monthly = (
            ai_df[ai_df[dim] == cat]
            .groupby(['job_posting_year', 'job_posting_month'])
            .size()
            .reset_index(name='count')
        )
        cat_monthly['date'] = pd.to_datetime(
            cat_monthly['job_posting_year'].astype(str) + '-'
            + cat_monthly['job_posting_month'].astype(str).str.zfill(2) + '-01'
        )
        cat_monthly = cat_monthly.set_index('date').reindex(monthly_total.index, fill_value=0)
        shares_df[cat] = cat_monthly['count'] / monthly_total['count']
    segment_shares[dim] = shares_df
    print(f"  {dim}: {len(categories)} categories, shares sum check = "
          f"{shares_df.sum(axis=1).mean():.4f}")

# --- 3. Skill rate series ---
skill_cols_forecast = ['skills_python', 'skills_sql', 'skills_ml',
                       'skills_deep_learning', 'skills_cloud']
skill_rates = pd.DataFrame(index=monthly_total.index)
for skill in skill_cols_forecast:
    monthly_rate = (
        ai_df.groupby(['job_posting_year', 'job_posting_month'])[skill]
        .mean()
        .reset_index(name='rate')
    )
    monthly_rate['date'] = pd.to_datetime(
        monthly_rate['job_posting_year'].astype(str) + '-'
        + monthly_rate['job_posting_month'].astype(str).str.zfill(2) + '-01'
    )
    monthly_rate = monthly_rate.set_index('date').reindex(monthly_total.index)
    skill_rates[skill] = monthly_rate['rate']
print(f"  Skills: {len(skill_cols_forecast)} rate series built")

# --- 4. Entry-level sub-segment shares (for Audience A dashboard) ---
entry_df_forecast = ai_df[ai_df['experience_level'] == 'Entry']
entry_sub_dims = {
    'remote_type': ['Remote', 'Hybrid', 'Onsite'],
    'company_industry': ai_df['company_industry'].unique().tolist(),
}
entry_monthly_total = (
    entry_df_forecast.groupby(['job_posting_year', 'job_posting_month'])
    .size()
    .reset_index(name='count')
)
entry_monthly_total['date'] = pd.to_datetime(
    entry_monthly_total['job_posting_year'].astype(str) + '-'
    + entry_monthly_total['job_posting_month'].astype(str).str.zfill(2) + '-01'
)
entry_monthly_total = entry_monthly_total.set_index('date').reindex(monthly_total.index, fill_value=0)

entry_segment_shares = {}
for dim, categories in entry_sub_dims.items():
    shares_df = pd.DataFrame(index=monthly_total.index)
    for cat in categories:
        cat_monthly = (
            entry_df_forecast[entry_df_forecast[dim] == cat]
            .groupby(['job_posting_year', 'job_posting_month'])
            .size()
            .reset_index(name='count')
        )
        cat_monthly['date'] = pd.to_datetime(
            cat_monthly['job_posting_year'].astype(str) + '-'
            + cat_monthly['job_posting_month'].astype(str).str.zfill(2) + '-01'
        )
        cat_monthly = cat_monthly.set_index('date').reindex(monthly_total.index, fill_value=0)
        shares_df[cat] = cat_monthly['count'] / entry_monthly_total['count'].replace(0, np.nan)
    shares_df = shares_df.fillna(1.0 / len(categories))  # uniform if no data
    entry_segment_shares[dim] = shares_df

# --- 5. Entry-level skill rates ---
entry_skill_rates = pd.DataFrame(index=monthly_total.index)
for skill in skill_cols_forecast:
    monthly_rate = (
        entry_df_forecast.groupby(['job_posting_year', 'job_posting_month'])[skill]
        .mean()
        .reset_index(name='rate')
    )
    monthly_rate['date'] = pd.to_datetime(
        monthly_rate['job_posting_year'].astype(str) + '-'
        + monthly_rate['job_posting_month'].astype(str).str.zfill(2) + '-01'
    )
    monthly_rate = monthly_rate.set_index('date').reindex(monthly_total.index)
    entry_skill_rates[skill] = monthly_rate['rate']

print("\nForecast data preparation complete.")
print(f"  Total series: {len(monthly_total)} months (Jan 2020 - Mar 2026)")
print(f"  Forecast horizon: 33 months (Apr 2026 - Dec 2028)")

Monthly total series: 75 months, 125-725 range
  experience_level: 3 categories, shares sum check = 1.0000
  remote_type: 3 categories, shares sum check = 1.0000
  company_industry: 6 categories, shares sum check = 1.0000
  country: 7 categories, shares sum check = 1.0000
  Skills: 5 rate series built

Forecast data preparation complete.
  Total series: 75 months (Jan 2020 - Mar 2026)
  Forecast horizon: 33 months (Apr 2026 - Dec 2028)


In [13]:
### Layer 1: Total Postings — SARIMA Forecast

FORECAST_STEPS = 33  # Apr 2026 - Dec 2028

# --- 1. Grid search for best SARIMA order ---
y = monthly_total['count'].astype(float)

best_aic = np.inf
best_order = None
best_seasonal = None
results_log = []

p_range = range(3)  # 0, 1, 2
d_range = range(2)  # 0, 1
q_range = range(3)
P_range = range(2)  # 0, 1
D_range = range(2)
Q_range = range(2)

print("Fitting SARIMA models (this may take a minute)...")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for p, d, q in product(p_range, d_range, q_range):
        for P, D, Q in product(P_range, D_range, Q_range):
            try:
                model = SARIMAX(y, order=(p, d, q),
                                seasonal_order=(P, D, Q, 12),
                                enforce_stationarity=False,
                                enforce_invertibility=False, trend='c')
                fit = model.fit(disp=False, maxiter=200)
                results_log.append({
                    'order': (p, d, q),
                    'seasonal': (P, D, Q, 12),
                    'aic': fit.aic
                })
                if fit.aic < best_aic:
                    best_aic = fit.aic
                    best_order = (p, d, q)
                    best_seasonal = (P, D, Q, 12)
            except Exception:
                continue

print(f"\nBest SARIMA{best_order}x{best_seasonal} — AIC: {best_aic:.1f}")
print(f"Models evaluated: {len(results_log)}")

# --- 2. Fit best model and forecast ---
best_model = SARIMAX(y, order=best_order, seasonal_order=best_seasonal,
                     enforce_stationarity=False, enforce_invertibility=False, trend='c')
sarima_fit = best_model.fit(disp=False)

forecast_result = sarima_fit.get_forecast(steps=FORECAST_STEPS)
forecast_mean = forecast_result.predicted_mean
forecast_ci_80 = forecast_result.conf_int(alpha=0.20)
forecast_ci_95 = forecast_result.conf_int(alpha=0.05)

# Build forecast date index
last_date = monthly_total.index[-1]
forecast_dates = pd.date_range(start=last_date + pd.DateOffset(months=1),
                               periods=FORECAST_STEPS, freq='MS')
forecast_mean.index = forecast_dates
forecast_ci_80.index = forecast_dates
forecast_ci_95.index = forecast_dates

# --- 3. Build combined historical + forecast DataFrame ---
total_forecast_df = pd.DataFrame({
    'date': list(monthly_total.index) + list(forecast_dates),
    'count': list(y.values) + list(forecast_mean.values),
    'is_forecast': [False] * len(y) + [True] * FORECAST_STEPS,
})
total_forecast_df['ci_80_lower'] = np.nan
total_forecast_df['ci_80_upper'] = np.nan
total_forecast_df['ci_95_lower'] = np.nan
total_forecast_df['ci_95_upper'] = np.nan
total_forecast_df.loc[total_forecast_df['is_forecast'], 'ci_80_lower'] = forecast_ci_80.iloc[:, 0].values
total_forecast_df.loc[total_forecast_df['is_forecast'], 'ci_80_upper'] = forecast_ci_80.iloc[:, 1].values
total_forecast_df.loc[total_forecast_df['is_forecast'], 'ci_95_lower'] = forecast_ci_95.iloc[:, 0].values
total_forecast_df.loc[total_forecast_df['is_forecast'], 'ci_95_upper'] = forecast_ci_95.iloc[:, 1].values

print(f"\nForecast range: {forecast_dates[0].strftime('%Y-%m')} to {forecast_dates[-1].strftime('%Y-%m')}")
print(f"Predicted monthly avg: {forecast_mean.mean():.0f} (historical avg: {y.mean():.0f})")
print(f"\nModel Summary:")
print(sarima_fit.summary().tables[1])

Fitting SARIMA models (this may take a minute)...

Best SARIMA(0, 1, 2)x(0, 1, 1, 12) — AIC: 513.8
Models evaluated: 144


/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)



Forecast range: 2026-04 to 2028-12
Predicted monthly avg: 507 (historical avg: 225)

Model Summary:
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept      1.4903      4.349      0.343      0.732      -7.034      10.014
ma.L1         -0.8818      0.313     -2.819      0.005      -1.495      -0.269
ma.L2          0.2774      0.496      0.559      0.576      -0.695       1.250
ma.S.L12      -0.4504      0.452     -0.997      0.319      -1.336       0.435
sigma2      2622.3948    419.039      6.258      0.000    1801.093    3443.697


/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [14]:
### Layer 2: Proportion Forecasts — Segment Composition Over Time

def fit_best_arima(series, max_p=2, max_d=1, max_q=2):
    """Fit best ARIMA by AIC. Returns (fitted_model, order) or (None, None) on failure."""
    best_aic = np.inf
    best_fit = None
    best_order = None
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for p, d, q in product(range(max_p + 1), range(max_d + 1), range(max_q + 1)):
            if p == 0 and q == 0 and d == 0:
                continue  # skip trivial model
            try:
                model = ARIMA(series, order=(p, d, q))
                fit = model.fit()
                if fit.aic < best_aic:
                    best_aic = fit.aic
                    best_fit = fit
                    best_order = (p, d, q)
            except Exception:
                continue
    return best_fit, best_order

# --- 1. Forecast shares for each segment dimension ---
segment_forecasts = {}  # {dim: DataFrame with forecast_dates index, one col per category}
model_log = []  # for Audience B summary table

print("Fitting proportion models...")
for dim, shares_df in segment_shares.items():
    forecast_shares = pd.DataFrame(index=forecast_dates)
    for cat in shares_df.columns:
        series = shares_df[cat].astype(float)
        fit, order = fit_best_arima(series)
        if fit is not None:
            fc = fit.forecast(steps=FORECAST_STEPS)
            forecast_shares[cat] = fc.values
            model_log.append({
                'dimension': dim, 'category': cat,
                'order': order, 'aic': fit.aic, 'type': 'share'
            })
        else:
            # Fallback: historical mean share
            forecast_shares[cat] = series.mean()
            model_log.append({
                'dimension': dim, 'category': cat,
                'order': 'fallback (mean)', 'aic': None, 'type': 'share'
            })
    # Normalize shares to sum to 1.0
    row_sums = forecast_shares.sum(axis=1)
    forecast_shares = forecast_shares.div(row_sums, axis=0)
    segment_forecasts[dim] = forecast_shares
    print(f"  {dim}: {len(shares_df.columns)} categories done, "
          f"shares sum = {forecast_shares.sum(axis=1).mean():.4f}")

# --- 2. Convert shares to counts ---
segment_count_forecasts = {}
for dim, fc_shares in segment_forecasts.items():
    segment_count_forecasts[dim] = fc_shares.multiply(forecast_mean.values, axis=0)

# --- 3. Build combined historical + forecast DataFrames per dimension ---
segment_combined = {}  # {dim: {cat: DataFrame with date, count, share, is_forecast}}
for dim in segment_shares:
    dim_data = {}
    for cat in segment_shares[dim].columns:
        hist_counts = (segment_shares[dim][cat] * monthly_total['count']).values
        hist_shares = segment_shares[dim][cat].values
        fc_counts = segment_count_forecasts[dim][cat].values
        fc_shares = segment_forecasts[dim][cat].values
        dim_data[cat] = pd.DataFrame({
            'date': list(monthly_total.index) + list(forecast_dates),
            'count': list(hist_counts) + list(fc_counts),
            'share': list(hist_shares) + list(fc_shares),
            'is_forecast': [False] * len(monthly_total) + [True] * FORECAST_STEPS,
        })
    segment_combined[dim] = dim_data

# --- 4. Entry-level sub-segment forecasts ---
# Get entry-level count forecast from Layer 2
entry_share_forecast = segment_forecasts['experience_level']['Entry']
entry_count_forecast = entry_share_forecast * forecast_mean.values
entry_count_historical = segment_shares['experience_level']['Entry'] * monthly_total['count']

entry_sub_forecasts = {}
for dim, shares_df in entry_segment_shares.items():
    fc_shares = pd.DataFrame(index=forecast_dates)
    for cat in shares_df.columns:
        series = shares_df[cat].astype(float)
        fit, order = fit_best_arima(series)
        if fit is not None:
            fc = fit.forecast(steps=FORECAST_STEPS)
            fc_shares[cat] = fc.values
            model_log.append({
                'dimension': f'entry_{dim}', 'category': cat,
                'order': order, 'aic': fit.aic, 'type': 'entry_share'
            })
        else:
            fc_shares[cat] = series.mean()
            model_log.append({
                'dimension': f'entry_{dim}', 'category': cat,
                'order': 'fallback (mean)', 'aic': None, 'type': 'entry_share'
            })
    row_sums = fc_shares.sum(axis=1)
    fc_shares = fc_shares.div(row_sums, axis=0)
    entry_sub_forecasts[dim] = fc_shares

model_log_df = pd.DataFrame(model_log)
print(f"\nProportion forecasts complete. {len(model_log)} models fitted.")
print(model_log_df[['dimension', 'category', 'order', 'aic']].to_string(index=False))

Fitting proportion models...


/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopmen

  experience_level: 3 categories done, shares sum = 1.0000


/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopmen

  remote_type: 3 categories done, shares sum = 1.0000


...

/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopmen

  country: 7 categories done, shares sum = 1.0000


/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopmen


Proportion forecasts complete. 28 models fitted.
             dimension   category     order         aic
      experience_level      Entry [1, 0, 1] -339.211181
      experience_level        Mid [0, 0, 2] -277.329814
      experience_level     Senior [1, 0, 0] -282.625869
           remote_type     Remote [1, 1, 0] -191.016385
           remote_type     Hybrid [2, 1, 2] -209.243305
           remote_type     Onsite [0, 1, 1] -263.029913
      company_industry Technology [1, 0, 1] -324.991670
      company_industry  Education [0, 0, 1] -320.504422
      company_industry    Finance [0, 0, 2] -336.748912
      company_industry E-commerce [1, 0, 0] -304.100265
      company_industry Healthcare [1, 0, 1] -319.711961
      company_industry     Retail [1, 0, 0] -329.179066
               country     Canada [0, 0, 1] -318.133292
               country  Australia [0, 0, 2] -344.675438
               country         UK [0, 0, 1] -375.585899
               country      India [0, 0, 2] -323.07214

/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [15]:
### Skills Rate Forecasts

# --- 1. Forecast overall skill requirement rates ---
skill_forecasts = pd.DataFrame(index=forecast_dates)
skill_model_log = []

print("Fitting skill rate models...")
for skill in skill_cols_forecast:
    series = skill_rates[skill].astype(float)
    fit, order = fit_best_arima(series)
    if fit is not None:
        fc = fit.forecast(steps=FORECAST_STEPS)
        skill_forecasts[skill] = fc.values.clip(0, 1)
        skill_model_log.append({
            'dimension': 'skill_rate', 'category': skill,
            'order': order, 'aic': fit.aic, 'type': 'skill'
        })
    else:
        skill_forecasts[skill] = series.mean()
        skill_model_log.append({
            'dimension': 'skill_rate', 'category': skill,
            'order': 'fallback (mean)', 'aic': None, 'type': 'skill'
        })

# --- 2. Forecast entry-level skill rates ---
entry_skill_forecasts = pd.DataFrame(index=forecast_dates)
for skill in skill_cols_forecast:
    series = entry_skill_rates[skill].astype(float)
    fit, order = fit_best_arima(series)
    if fit is not None:
        fc = fit.forecast(steps=FORECAST_STEPS)
        entry_skill_forecasts[skill] = fc.values.clip(0, 1)
        skill_model_log.append({
            'dimension': 'entry_skill_rate', 'category': skill,
            'order': order, 'aic': fit.aic, 'type': 'entry_skill'
        })
    else:
        entry_skill_forecasts[skill] = series.mean()
        skill_model_log.append({
            'dimension': 'entry_skill_rate', 'category': skill,
            'order': 'fallback (mean)', 'aic': None, 'type': 'entry_skill'
        })

# --- 3. Build combined skill DataFrames ---
skill_combined = pd.DataFrame(index=list(monthly_total.index) + list(forecast_dates))
for skill in skill_cols_forecast:
    skill_combined[skill] = list(skill_rates[skill].values) + list(skill_forecasts[skill].values)
skill_combined['is_forecast'] = [False] * len(monthly_total) + [True] * FORECAST_STEPS

entry_skill_combined = pd.DataFrame(index=list(monthly_total.index) + list(forecast_dates))
for skill in skill_cols_forecast:
    entry_skill_combined[skill] = (
        list(entry_skill_rates[skill].values)
        + list(entry_skill_forecasts[skill].values)
    )
entry_skill_combined['is_forecast'] = [False] * len(monthly_total) + [True] * FORECAST_STEPS

# --- 4. Append to model log ---
skill_log_df = pd.DataFrame(skill_model_log)
all_model_log = pd.concat([model_log_df, skill_log_df], ignore_index=True)

print(f"\nSkill rate forecasts complete. {len(skill_model_log)} models fitted.")
for _, row in skill_log_df.iterrows():
    print(f"  {row['category']}: ARIMA{row['order']} (AIC={row['aic']:.1f})"
          if row['aic'] is not None
          else f"  {row['category']}: fallback (mean)")
print(f"\nTotal models across all dimensions: {len(all_model_log)}")

Fitting skill rate models...


/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopmen


Skill rate forecasts complete. 10 models fitted.
  skills_python: ARIMA[2, 0, 2] (AIC=-331.8)
  skills_sql: ARIMA[2, 0, 2] (AIC=-313.9)
  skills_ml: ARIMA[0, 0, 2] (AIC=-321.4)
  skills_deep_learning: ARIMA[1, 0, 0] (AIC=-312.5)
  skills_cloud: ARIMA[1, 0, 0] (AIC=-287.9)
  skills_python: ARIMA[0, 0, 1] (AIC=-201.7)
  skills_sql: ARIMA[0, 0, 2] (AIC=-196.7)
  skills_ml: ARIMA[2, 0, 1] (AIC=-155.8)
  skills_deep_learning: ARIMA[0, 0, 1] (AIC=-169.0)
  skills_cloud: ARIMA[1, 0, 0] (AIC=-208.1)

Total models across all dimensions: 38


/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/home/wanderduck/000_Duckspace/WanderduckDevelopment/Ducks/UMN/CERT-x466-003/.venv/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [16]:
### Diagnostics & Model Summary

# --- 1. Residual diagnostics for the total SARIMA model ---
residuals = sarima_fit.resid

# Ljung-Box test for residual autocorrelation
lb_test = acorr_ljungbox(residuals, lags=[12], return_df=True)
lb_pval = lb_test['lb_pvalue'].values[0]

print("=== Total SARIMA Model Diagnostics ===")
print(f"Model: SARIMA{best_order}x{best_seasonal}")
print(f"AIC: {sarima_fit.aic:.1f}")
print(f"Residual mean: {residuals.mean():.2f}")
print(f"Residual std: {residuals.std():.2f}")
print(f"Ljung-Box p-value (lag 12): {lb_pval:.4f} "
      f"({'no significant autocorrelation' if lb_pval > 0.05 else 'WARNING: residual autocorrelation detected'})")

# --- 2. Diagnostics plot (2x2) ---
fig_diag = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Residuals Over Time', 'ACF of Residuals',
                    'Residual Distribution', 'Q-Q Plot')
)

# Residuals over time
fig_diag.add_trace(go.Scatter(
    x=monthly_total.index, y=residuals,
    mode='lines', name='Residuals', line=dict(color='#636EFA')
), row=1, col=1)
fig_diag.add_hline(y=0, line_dash='dash', line_color='red', row=1, col=1)

# ACF
from statsmodels.tsa.stattools import acf
acf_vals, acf_ci = acf(residuals, nlags=24, alpha=0.05)
fig_diag.add_trace(go.Bar(
    x=list(range(25)), y=acf_vals,
    name='ACF', marker_color='#636EFA'
), row=1, col=2)
# Confidence bounds
ci_upper = 1.96 / np.sqrt(len(residuals))
fig_diag.add_hline(y=ci_upper, line_dash='dash', line_color='red', row=1, col=2)
fig_diag.add_hline(y=-ci_upper, line_dash='dash', line_color='red', row=1, col=2)

# Histogram
fig_diag.add_trace(go.Histogram(
    x=residuals, nbinsx=20, name='Residuals',
    marker_color='#636EFA'
), row=2, col=1)

# Q-Q plot
sorted_resid = np.sort(residuals)
theoretical_q = sp_stats.norm.ppf(np.linspace(0.01, 0.99, len(sorted_resid)))
fig_diag.add_trace(go.Scatter(
    x=theoretical_q, y=sorted_resid,
    mode='markers', name='Q-Q', marker=dict(color='#636EFA', size=4)
), row=2, col=2)
qq_min = min(theoretical_q.min(), sorted_resid.min())
qq_max = max(theoretical_q.max(), sorted_resid.max())
fig_diag.add_trace(go.Scatter(
    x=[qq_min, qq_max], y=[qq_min, qq_max],
    mode='lines', name='Reference', line=dict(color='red', dash='dash')
), row=2, col=2)

fig_diag.update_layout(
    title='SARIMA Model Diagnostics — Total Monthly Postings',
    height=600, showlegend=False
)

fig_diag.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig_diag.write_image('notebook_images/plot_images/new/forecast_diag.png', scale=2)
fig_diag.show()

# --- 3. Model summary table ---
print("\n=== All Fitted Models ===")
summary_display = all_model_log.copy()
summary_display['order'] = summary_display['order'].astype(str)
summary_display['aic'] = summary_display['aic'].apply(
    lambda x: f"{x:.1f}" if pd.notna(x) else "N/A"
)
print(summary_display[['dimension', 'category', 'type', 'order', 'aic']]
      .to_string(index=False))

=== Total SARIMA Model Diagnostics ===
Model: SARIMA(0, 1, 2)x(0, 1, 1, 12)
AIC: 513.8
Residual mean: 0.55
Residual std: 93.70
Ljung-Box p-value (lag 12): 0.0000 (WARNING: residual autocorrelation detected)



=== All Fitted Models ===
             dimension             category        type     order    aic
      experience_level                Entry       share [1, 0, 1] -339.2
      experience_level                  Mid       share [0, 0, 2] -277.3
      experience_level               Senior       share [1, 0, 0] -282.6
           remote_type               Remote       share [1, 1, 0] -191.0
           remote_type               Hybrid       share [2, 1, 2] -209.2
           remote_type               Onsite       share [0, 1, 1] -263.0
      company_industry           Technology       share [1, 0, 1] -325.0
      company_industry            Education       share [0, 0, 1] -320.5
      company_industry              Finance       share [0, 0, 2] -336.7
      company_industry           E-commerce       share [1, 0, 0] -304.1
      company_industry           Healthcare       share [1, 0, 1] -319.7
      company_industry               Retail       share [1, 0, 0] -329.2
               country  

In [23]:
### Audience A: Storytelling Visualizations
BOUNDARY_DATE = pd.Timestamp('2026-04-01').timestamp() * 1000  # Apr 2026

# ========================================================================
# Chart F1: Total Market Forecast
# ========================================================================
fig1 = go.Figure()

# Historical
hist = total_forecast_df[~total_forecast_df['is_forecast']]
fig1.add_trace(go.Scatter(
    x=hist['date'], y=hist['count'],
    mode='lines', name='Historical',
    line=dict(color='#636EFA', width=2)
))

# Forecast
fc = total_forecast_df[total_forecast_df['is_forecast']]
fig1.add_trace(go.Scatter(
    x=fc['date'], y=fc['count'],
    mode='lines', name='Forecast',
    line=dict(color='#636EFA', width=2, dash='dash')
))

# 95% CI band
fig1.add_trace(go.Scatter(
    x=list(fc['date']) + list(fc['date'][::-1]),
    y=list(fc['ci_95_upper']) + list(fc['ci_95_lower'][::-1]),
    fill='toself', fillcolor='rgba(99,110,250,0.1)',
    line=dict(color='rgba(0,0,0,0)'), name='95% CI', showlegend=True
))

# 80% CI band
fig1.add_trace(go.Scatter(
    x=list(fc['date']) + list(fc['date'][::-1]),
    y=list(fc['ci_80_upper']) + list(fc['ci_80_lower'][::-1]),
    fill='toself', fillcolor='rgba(99,110,250,0.2)',
    line=dict(color='rgba(0,0,0,0)'), name='80% CI', showlegend=True
))

# Boundary line
fig1.add_vline(x=BOUNDARY_DATE, line_dash='dot', line_color='gray',
               annotation_text='Forecast →', annotation_position='top right')

fig1.update_layout(
    title='Total AI Job Postings: Historical and Projected Through 2028',
    xaxis_title='Date', yaxis_title='Monthly Job Postings',
    height=450, template='plotly_white'
)

fig1.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig1.write_image('notebook_images/plot_images/new/forecast01.png', scale=2)
fig1.show()

# ========================================================================
# Chart F2: The Shifting Mix — Experience Level Composition
# ========================================================================
fig2 = go.Figure()

exp_colors = {'Entry': '#EF553B', 'Mid': '#636EFA', 'Senior': '#00CC96'}
for level in ['Senior', 'Mid', 'Entry']:  # stack order: bottom to top
    data = segment_combined['experience_level'][level]
    hist_data = data[~data['is_forecast']]
    fc_data = data[data['is_forecast']]

    # Historical shares
    fig2.add_trace(go.Scatter(
        x=hist_data['date'], y=hist_data['share'] * 100,
        mode='lines', name=f'{level} (historical)',
        line=dict(color=exp_colors[level], width=2),
        stackgroup='hist'
    ))
    # Forecast shares
    fig2.add_trace(go.Scatter(
        x=fc_data['date'], y=fc_data['share'] * 100,
        mode='lines', name=f'{level} (forecast)',
        line=dict(color=exp_colors[level], width=2, dash='dash'),
        stackgroup='fc', opacity=0.6
    ))

fig2.add_vline(x=BOUNDARY_DATE, line_dash='dot', line_color='gray')

fig2.update_layout(
    title='Experience Level Composition: Who Gets Hired? (% of Total Postings)',
    xaxis_title='Date', yaxis_title='Share of Postings (%)',
    height=500, template='plotly_white'
)

fig2.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig2.write_image('notebook_images/plot_images/new/forecast02.png', scale=2)
fig2.show()

# ========================================================================
# Chart F3: Entry-Level Outlook Dashboard (2x2)
# ========================================================================
fig3 = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Entry-Level: Remote Type Mix',
        'Entry-Level: Top Industries',
        'Entry-Level: Skills Demand Trajectory',
        'Entry-Level: Projected Posting Count'
    ),
    vertical_spacing=0.12, horizontal_spacing=0.08
)

# --- F3 Top-Left: Remote type shares for entry-level ---
remote_colors = {'Remote': '#636EFA', 'Hybrid': '#EF553B', 'Onsite': '#00CC96'}
for rtype in ['Remote', 'Hybrid', 'Onsite']:
    hist_shares = entry_segment_shares['remote_type'][rtype]
    fc_shares = entry_sub_forecasts['remote_type'][rtype]
    fig3.add_trace(go.Scatter(
        x=monthly_total.index, y=hist_shares * 100,
        mode='lines', name=rtype,
        line=dict(color=remote_colors[rtype], width=1.5),
        legendgroup=rtype, showlegend=True
    ), row=1, col=1)
    fig3.add_trace(go.Scatter(
        x=forecast_dates, y=fc_shares * 100,
        mode='lines', name=f'{rtype} (fc)',
        line=dict(color=remote_colors[rtype], width=1.5, dash='dash'),
        legendgroup=rtype, showlegend=False
    ), row=1, col=1)

# --- F3 Top-Right: Industry shares for entry-level ---
industry_list = entry_segment_shares['company_industry'].columns.tolist()
for ind in industry_list:
    hist_shares = entry_segment_shares['company_industry'][ind]
    fc_shares = entry_sub_forecasts['company_industry'][ind]
    fig3.add_trace(go.Scatter(
        x=monthly_total.index, y=hist_shares * 100,
        mode='lines', name=ind,
        line=dict(width=1.5),
        legendgroup=ind, showlegend=True
    ), row=1, col=2)
    fig3.add_trace(go.Scatter(
        x=forecast_dates, y=fc_shares * 100,
        mode='lines', name=f'{ind} (fc)',
        line=dict(width=1.5, dash='dash'),
        legendgroup=ind, showlegend=False
    ), row=1, col=2)

# --- F3 Bottom-Left: Entry-level skill rates ---
skill_labels = {
    'skills_python': 'Python', 'skills_sql': 'SQL', 'skills_ml': 'ML',
    'skills_deep_learning': 'Deep Learning', 'skills_cloud': 'Cloud'
}
skill_colors = {
    'skills_python': '#636EFA', 'skills_sql': '#EF553B', 'skills_ml': '#00CC96',
    'skills_deep_learning': '#AB63FA', 'skills_cloud': '#FFA15A'
}
for skill in skill_cols_forecast:
    label = skill_labels[skill]
    color = skill_colors[skill]
    hist_vals = entry_skill_combined[~entry_skill_combined['is_forecast']][skill]
    fc_vals = entry_skill_combined[entry_skill_combined['is_forecast']][skill]
    fig3.add_trace(go.Scatter(
        x=monthly_total.index, y=hist_vals * 100,
        mode='lines', name=label,
        line=dict(color=color, width=1.5),
        legendgroup=label, showlegend=True
    ), row=2, col=1)
    fig3.add_trace(go.Scatter(
        x=forecast_dates, y=fc_vals * 100,
        mode='lines', name=f'{label} (fc)',
        line=dict(color=color, width=1.5, dash='dash'),
        legendgroup=label, showlegend=False
    ), row=2, col=1)

# --- F3 Bottom-Right: Entry-level posting count ---
fig3.add_trace(go.Scatter(
    x=monthly_total.index, y=entry_count_historical.values,
    mode='lines', name='Entry Count',
    line=dict(color='#EF553B', width=2),
    showlegend=False
), row=2, col=2)
fig3.add_trace(go.Scatter(
    x=forecast_dates, y=entry_count_forecast.values,
    mode='lines', name='Entry Count (fc)',
    line=dict(color='#EF553B', width=2, dash='dash'),
    showlegend=False
), row=2, col=2)

# Add boundary lines to all subplots
for row in [1, 2]:
    for col in [1, 2]:
        fig3.add_vline(x=BOUNDARY_DATE, line_dash='dot', line_color='gray',
                       row=row, col=col)

fig3.update_layout(
    title='Entry-Level Outlook: Remote Work, Industries, Skills, and Volume Through 2028',
    height=700, template='plotly_white',
    legend=dict(font=dict(size=9))
)
fig3.update_yaxes(title_text='% of Entry Postings', row=1, col=1)
fig3.update_yaxes(title_text='% of Entry Postings', row=1, col=2)
fig3.update_yaxes(title_text='% Requiring Skill', row=2, col=1)
fig3.update_yaxes(title_text='Monthly Postings', row=2, col=2)

fig3.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.69)'),
    yaxis=dict(gridcolor='rgba(255,255,255,0.69)')
)

fig3.write_image('notebook_images/plot_images/new/forecast03.png', scale=2)
fig3.show()

### Methodology & Limitations (Technical Appendix)

#### Forecasting Approach

This forecast uses a **hybrid aggregate + proportion** architecture:

1. **Layer 1 (Total Volume):** A Seasonal ARIMA (SARIMA) model fitted on 84 months of aggregate monthly posting counts (Jan 2020 – Dec 2026). Model order was selected by AIC grid search over (p,d,q)(P,D,Q,12) with p,q ∈ {0,1,2}, P,Q ∈ {0,1}, d,D ∈ {0,1}. The model produces point forecasts with 80% and 95% prediction intervals for 24 months (Jan 2027 – Dec 2028).

2. **Layer 2 (Compositional Shifts):** For each segment dimension (experience level, remote type, industry, country), monthly category shares are modeled with per-category ARIMA. Forecast shares are normalized to sum to 1.0 at each month, then multiplied by the Layer 1 total forecast to produce segment-level count projections.

3. **Skills (Rate Models):** Binary skill columns are aggregated to monthly rates (proportion of postings requiring each skill) and forecast independently with ARIMA. Forecasts are clamped to [0, 1].

#### Key Limitations

- **Synthetic data.** This dataset was generated, not collected from real job postings. The near-uniform distribution across categories (e.g., ~5,200 postings per experience level, ~2,200 per country) is a clear synthetic signature. Real job market data would show much more variance, stronger trends, and structural breaks (e.g., COVID-19 impacts, the 2023 tech layoffs). **Forecasts from this data should be interpreted as methodological demonstrations, not real-world predictions.**

- **Stationary totals.** Yearly posting counts range 2,155–2,309 with no meaningful growth trend. The SARIMA model correctly identifies this stationarity — the forecast is essentially “more of the same” with widening uncertainty. This is honest modeling, not a failure.

- **Small cell sizes at granular levels.** Country-by-month segments average ~26 observations. Share estimates at this level are noisy, and ARIMA models on noisy share series may converge to near-constant forecasts (historical mean). Where a model fails to converge, the historical mean is used as a fallback.

- **Simplified uncertainty.** Segment-level prediction intervals are derived proportionally from the total forecast’s intervals. Full uncertainty propagation (combining share model uncertainty with total model uncertainty) would be more rigorous but adds complexity for minimal practical gain given the data characteristics.

- **Independence assumption.** Segment dimensions are modeled independently — the model does not capture interactions (e.g., “entry-level remote roles in tech” as a distinct segment). With 84 months of data and synthetic uniformity, interaction modeling would overfit.